# M11.5 — Real off-source detector noise with frozen M10

## Objective

M11.4 isolated the effect of replacing the analytical M10 PSD by an empirical detector PSD while keeping the stochastic noise Gaussian.

We now introduce actual real off-source detector strain.

The two domains of interest are

$$
G1 =
\text{Gaussian noise generated from an empirical PSD},
$$

and

$$
R =
\text{actual real off-source detector strain}.
$$

For a given real-noise epoch, both domains use the same empirical PSD for:

- optimal-SNR definition;
- signal-distance scaling;
- whitening and preprocessing.

Therefore the comparison

$$
G1 \rightarrow R
$$

is designed to isolate detector-noise properties that cannot be fully described by the PSD alone.

These may include:

- non-Gaussian fluctuations;
- non-stationarity;
- transient disturbances;
- imperfectly stationary spectral lines;
- correlations and temporal structure beyond the second-order PSD model.

The M10 predictor remains frozen.

No retraining is performed in M11.5.

## Main hypotheses

### H1 — Input-domain shift

Even when G1 and R share the same empirical PSD, the final processed inputs may differ because real strain is not an exact stationary Gaussian realization of that PSD.

### H2 — Latent-domain shift

The frozen M10 embedding may move more strongly under R than under the G0-to-G1 PSD-only transition studied in M11.4.

### H3 — Prediction sensitivity

Real-noise structure may increase prediction variability or bias relative to the Gaussian empirical-PSD control.

### H4 — Realized-SNR distribution

For G1, the fixed-template statistic should remain approximately

$$
\Delta\rho
=
\rho_{\rm real}-\rho_{\rm opt}
\sim
\mathcal N(0,1).
$$

For R, deviations in variance, asymmetry or tails are allowed and become a diagnostic of real-noise behavior beyond the Gaussian PSD model.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ks_2samp, wasserstein_distance

from pycbc.types import TimeSeries, FrequencySeries

PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.real_data.psd import (
    estimate_offsource_psd,
    psd_window_is_available_and_finite,
)

from src.real_data.gwosc_utils import (
    read_gwosc_hdf5_as_pycbc_timeseries,
)

from src.config import SimulationConfig
from src.dataset import DatasetBuilder
from src.parameters import CBCParameters
from src.snr import compute_network_optimal_snr

## 1. Experimental setup and real-noise region

In [ ]:
EVENT_NAME = "GW170814"
EVENT_TIME = 1186741861.5

DETECTORS = ["H1", "L1", "V1"]

PSD_WINDOW_R = (-1024.0, -640.0)
PSD_SEGMENT_DURATION_R = 8.0

In [ ]:
REAL_NOISE_REGION_R = (-512.0, -128.0)

REAL_CENTER_SPACING_S = 12.0 # Space between inputs (our context is 4.8s), to avoid overlap and too-close examples.
N_R_MAX = 30

In [ ]:
region_start = (
    EVENT_TIME
    + REAL_NOISE_REGION_R[0]
)

region_end = (
    EVENT_TIME
    + REAL_NOISE_REGION_R[1]
)

candidate_centers_R = np.arange(
    region_start + 8.0,
    region_end - 8.0,
    REAL_CENTER_SPACING_S,
)

candidate_centers_R = (
    candidate_centers_R[:N_R_MAX]
)

print(len(candidate_centers_R))
print(candidate_centers_R[:5])

The real-noise region is deliberately separated from the PSD-estimation window.

The empirical PSD is estimated from

$$
[-1024,-640]\ {\rm s}
$$

relative to GW170814, whereas real-noise injection centers are selected from

$$
[-512,-128]\ {\rm s}.
$$

This avoids using the exact strain segment under test to estimate the noise model used for whitening and optimal-SNR control.

Candidate centers are separated by 12 s, which is substantially larger than the approximately 4.8 s processing window. This reduces overlap and correlation between neighboring real-noise examples.

In [ ]:
config = SimulationConfig(
    sampling_frequency=4096.0,
    duration=4.0,
    low_frequency_cutoff=30.0,
    waveform_approximant="SEOBNRv4_opt",

    # Para esta primera prueba prefiero NO reescalar distancia
    target_network_snr_range=None,

    processing_context_start_samples=1664,
    processing_context_end_samples=1664,
)

In [ ]:
builder = DatasetBuilder.from_config(
    config=config,
    detector_names=DETECTORS,
    signal_processor_kwargs={
        "whitening_method": "psd",
        "apply_highpass": True,
        "apply_lowpass": True,
        "apply_standardization": False,
        "output_mode": "crop_to_config",
        "whitening_low_frequency_cutoff": 30.0,
        "whitening_max_filter_duration": 0.5,
        "whitening_trunc_method": "hann",
        "highpass_frequency": 30.0,
        "lowpass_frequency": 512.0,
        "fir_order": 256,
        "fir_beta": 5.0,
        "remove_corrupted": True,
    },
    rng=np.random.default_rng(123),
)

In [ ]:
injector = builder.signal_injector
processor = builder.signal_processor
noise_model = builder.noise_model

print(type(injector).__name__)
print(type(processor).__name__)
print(type(noise_model).__name__)

## 2. Load and validate GWOSC strain

GW170814 is retained as the initial M11.5 benchmark because its H1, L1 and V1 strain data and off-source PSD workflow were already validated during the previous M10 and M11 analyses.

At this stage the astrophysical GW170814 signal itself is not used.

Only off-source detector strain surrounding the event is used to construct controlled real-noise backgrounds.

In [ ]:
from src.real_data.catalog import (fetch_gwosc_catalog_events, 
                                   gwosc_events_to_parameter_df,
                                   build_gwosc_urls_for_events)

from src.real_data.gwosc_utils import (download_if_needed)

In [ ]:
GWOSC_CATALOG = "GWTC-1-confident"

gwosc_events = fetch_gwosc_catalog_events(
    catalog=GWOSC_CATALOG,
    include_default_parameters=True,
)

gwosc_catalog_df = gwosc_events_to_parameter_df(
    gwosc_events,
    catalog_name=GWOSC_CATALOG,
)

GWOSC_URLS_ACTIVE, gwosc_url_failures_df = (
    build_gwosc_urls_for_events(
        gwosc_catalog_df,
        catalog=GWOSC_CATALOG,
        sample_rate=4096,
    )
)

print("Number of active events:", len(GWOSC_URLS_ACTIVE))
print("GW170814 available:", "GW170814" in GWOSC_URLS_ACTIVE)

In [ ]:
EVENT_NAME = "GW170814"

if EVENT_NAME not in GWOSC_URLS_ACTIVE:
    raise KeyError(
        f"{EVENT_NAME} not found in catalog {GWOSC_CATALOG}. "
        f"Available examples: {list(GWOSC_URLS_ACTIVE)[:10]}"
    )

GWOSC_CACHE_DIR = Path("gwosc_cache")
GWOSC_CACHE_DIR.mkdir(exist_ok=True)

In [ ]:
# We filter those events without a complete H1-L1-V1 set

event_urls = GWOSC_URLS_ACTIVE[EVENT_NAME]

missing = [
    det
    for det in DETECTORS
    if det not in event_urls
]

if missing:
    raise ValueError(
        f"{EVENT_NAME} does not provide all required detectors. "
        f"Missing: {missing}. "
        f"Available: {list(event_urls)}"
    )

In [ ]:
raw_strains_long = {}

for det in DETECTORS:

    url = event_urls[det]

    local_path = download_if_needed(
        url,
        cache_dir=GWOSC_CACHE_DIR,
    )

    raw_strains_long[det] = (
        read_gwosc_hdf5_as_pycbc_timeseries(
            local_path
        )
    )

In [ ]:
for ifo, strain in raw_strains_long.items():

    print(
        ifo,
        len(strain),
        float(strain.start_time),
        float(strain.end_time),
        float(strain.delta_t),
    )

In [ ]:
def real_processing_slice_is_valid(
    raw_strains,
    center_time,
):
    start = (
        center_time
        - config.duration / 2
        - config.processing_context_start_seconds
    )

    end = (
        center_time
        + config.duration / 2
        + config.processing_context_end_seconds
    )

    for ifo in DETECTORS:

        ts = raw_strains[ifo]

        if (
            start < float(ts.start_time)
            or end > float(ts.end_time)
        ):
            return False

        segment = ts.time_slice(
            start,
            end,
        )

        if len(segment) != config.processing_length:
            return False

        if not np.all(
            np.isfinite(
                segment.numpy()
            )
        ):
            return False

    return True

In [ ]:
valid_centers_R = [
    float(t)
    for t in candidate_centers_R
    if real_processing_slice_is_valid(
        raw_strains_long,
        t,
    )
]

print(
    "valid centers:",
    len(valid_centers_R),
)

## 3. Empirical PSD shared by G1 and R

For the initial M11.5 experiment, one empirical PSD window is kept fixed for all selected real-noise epochs.

The same PSD is used in both domains:

$$
S_n^{G1}(f)=S_n^{R}(f).
$$

It controls:

- optimal-SNR evaluation;
- distance rescaling;
- Gaussian G1 generation;
- whitening of both G1 and R.

This keeps the PSD model fixed while changing the stochastic noise process itself.

In [ ]:
empirical_psds_processing = {}

for ifo in DETECTORS:

    empirical_psds_processing[ifo] = (
        estimate_offsource_psd(
            strain=raw_strains_long[ifo],
            event_time=EVENT_TIME,
            delta_f=config.processing_delta_f,
            sampling_frequency=config.sampling_frequency,
            target_flength=config.processing_flength,
            psd_start_offset=PSD_WINDOW_R[0],
            psd_end_offset=PSD_WINDOW_R[1],
            psd_segment_duration=PSD_SEGMENT_DURATION_R,
            low_frequency_cutoff=config.low_frequency_cutoff,
            max_filter_duration=0.5,
            trunc_method="hann",
        )
    )

In [ ]:
empirical_psds_4s = {}

for ifo in DETECTORS:

    empirical_psds_4s[ifo] = (
        estimate_offsource_psd(
            strain=raw_strains_long[ifo],
            event_time=EVENT_TIME,
            delta_f=config.delta_f,
            sampling_frequency=config.sampling_frequency,
            target_flength=config.flength,
            psd_start_offset=PSD_WINDOW_R[0],
            psd_end_offset=PSD_WINDOW_R[1],
            psd_segment_duration=PSD_SEGMENT_DURATION_R,
            low_frequency_cutoff=config.low_frequency_cutoff,
            max_filter_duration=0.5,
            trunc_method="hann",
        )
    )

## 4. Fixed source and epoch-dependent detector projection

The intrinsic source parameters are held fixed across all epochs.

For every real-noise center, however, the GW is projected using

$$
t_{\rm geo}=t_{\rm GPS,center}.
$$

Therefore antenna responses and inter-detector delays remain physically consistent with the actual detector epoch.

The luminosity distance is rescaled independently at each epoch to enforce

$$
\rho_{\rm net,opt}=15
$$

under the empirical PSD.

In [ ]:
BASE_PARAMS_R = CBCParameters(
    mass_1=30.0,
    mass_2=30.0,
    distance=500.0,
    inclination=1.0,
    ra=1.0,
    dec=0.3,
    spin_1z=0.0,
    spin_2z=0.0,
    polarization_angle=0.5,
)

TARGET_RHO_R = 15.0

In [ ]:
def build_network_at_real_epoch(
    center_time,
    target_rho=15.0,
):
    net0 = builder._build_projected_signal_network(
        params=BASE_PARAMS_R,
        geocentric_coalescence_time=center_time,
        placement_policy="centered",
    )

    _, rho0 = compute_network_optimal_snr(
        signal_segments=net0.signal_segments,
        psds=empirical_psds_4s,
        config=config,
    )

    distance_target = (
        BASE_PARAMS_R.distance
        * rho0
        / target_rho
    )

    params_target = (
        BASE_PARAMS_R.with_distance(
            distance_target
        )
    )

    net = builder._build_projected_signal_network(
        params=params_target,
        geocentric_coalescence_time=center_time,
        placement_policy="centered",
    )

    det_snr, rho = compute_network_optimal_snr(
        signal_segments=net.signal_segments,
        psds=empirical_psds_4s,
        config=config,
    )

    return {
        "network": net,
        "detector_snrs": det_snr,
        "network_snr": rho,
        "distance": distance_target,
    }

In [ ]:
test_epoch_R = valid_centers_R[0]

test_net_R = build_network_at_real_epoch(
    test_epoch_R,
)

print(
    "GPS:",
    test_epoch_R,
)

print(
    "distance:",
    test_net_R["distance"],
)

print(
    "rho:",
    test_net_R["network_snr"],
)

print(
    "detector SNR:",
    test_net_R["detector_snrs"],
)

In [ ]:
assert np.isclose(
    test_net_R["network_snr"],
    TARGET_RHO_R,
    rtol=1e-6,
)

assert np.all(
    np.isfinite(
        list(
            test_net_R["detector_snrs"].values()
        )
    )
)

print("Epoch-dependent network contract passed.")

## 5. Extract the real processing-context strain

In [ ]:
def extract_real_processing_noise(
    raw_strains,
    network,
):
    output_start = float(
        network.placement.segment_start_time
    )

    context_start = (
        output_start
        - config.processing_context_start_seconds
    )

    context_end = (
        output_start
        + config.duration
        + config.processing_context_end_seconds
    )

    noises = {}

    for ifo in DETECTORS:

        segment = raw_strains[ifo].time_slice(
            context_start,
            context_end,
        )

        if len(segment) != config.processing_length:
            raise ValueError(
                f"{ifo}: unexpected real-noise length "
                f"{len(segment)}"
            )

        if not np.all(
            np.isfinite(
                segment.numpy()
            )
        ):
            raise ValueError(
                f"{ifo}: non-finite real strain."
            )

        noises[ifo] = segment.copy()

    return noises

## 6. Inject the full projected GW into real strain

The M11.1 full-projection convention is used.

The complete projected physical waveform is injected into the longer processing-context segment, and only the portion overlapping the available strain is added.

The absolute GPS timing of both the real strain and projected signal is preserved.

In [ ]:
real_noise_test = (
    extract_real_processing_noise(
        raw_strains_long,
        test_net_R["network"],
    )
)

In [ ]:
for ifo in DETECTORS:

    noise = real_noise_test[ifo]
    signal = (
        test_net_R["network"]
        .projection.strains[ifo]
    )

    print(
        ifo,
        "noise:",
        float(noise.start_time),
        float(noise.end_time),
        "signal:",
        float(signal.start_time),
        float(signal.end_time),
    )

In [ ]:
injected_R_test = (
    injector.inject_network(
        noises=real_noise_test,
        signals=(
            test_net_R[
                "network"
            ].projection.strains
        ),
    )
)

In [ ]:
for ifo in DETECTORS:

    r = injected_R_test[ifo]

    print(
        ifo,
        "n_signal:",
        r.n_signal_samples,
        "n_injected:",
        r.n_injected_samples,
        "clipped_before:",
        r.n_clipped_before,
        "clipped_after:",
        r.n_clipped_after,
    )

## 7. Process the real-noise injection

In [ ]:
strains_R_test = {
    ifo: injected_R_test[ifo].strain
    for ifo in DETECTORS
}

In [ ]:
processed_R_test = (
    processor.process_network(
        strains=strains_R_test,
        psds=empirical_psds_processing,
    )
)

In [ ]:
#Stack

X_R_test = np.stack(
    [
        np.asarray(
            processed_R_test[ifo],
            dtype=np.float64,
        )
        for ifo in DETECTORS
    ],
    axis=0,
)

## M10 contract

In [ ]:
M10_INPUT_NORM_CONFIG = {
    "enabled": True,
    "mode": "per_sample_per_detector_zscore",
    "eps": 1e-6,
}

In [ ]:
def m10_input_zscore(X, config_norm):
    """
    Exact M10 per-sample/per-detector z-score.

    Parameters
    ----------
    X : np.ndarray
        Shape (n_detectors, n_samples).

    config_norm : dict
        M10 input-normalization configuration.

    Returns
    -------
    Z : np.ndarray
        Normalized input with same shape as X.
    means : np.ndarray
        Per-detector means.
    stds : np.ndarray
        Per-detector standard deviations.
    """
    X = np.asarray(
        X,
        dtype=np.float64,
    )

    if not config_norm["enabled"]:
        return X.copy(), None, None

    if (
        config_norm["mode"]
        != "per_sample_per_detector_zscore"
    ):
        raise ValueError(
            "Unsupported input normalization mode: "
            f"{config_norm['mode']}"
        )

    eps = float(
        config_norm["eps"]
    )

    means = X.mean(
        axis=1
    )

    stds = X.std(
        axis=1
    )

    if np.any(
        ~np.isfinite(means)
    ):
        raise ValueError(
            "Non-finite channel means."
        )

    if np.any(
        ~np.isfinite(stds)
    ):
        raise ValueError(
            "Non-finite channel stds."
        )

    if np.any(
        stds < eps
    ):
        raise ValueError(
            f"Input std below eps={eps}: "
            f"{stds}"
        )

    Z = (
        X
        - means[:, None]
    ) / stds[:, None]

    return Z, means, stds

In [ ]:
Z_R_test, _, _ = m10_input_zscore(
    X_R_test,
    M10_INPUT_NORM_CONFIG,
)

Z_R_test = Z_R_test.astype(
    np.float32
)

In [ ]:
print(
    X_R_test.shape,
    Z_R_test.shape,
)

print(
    "raw std:",
    X_R_test.std(axis=1),
)

print(
    "z mean:",
    Z_R_test.mean(axis=1),
)

print(
    "z std:",
    Z_R_test.std(axis=1),
)

In [ ]:
assert X_R_test.shape == (
    len(DETECTORS),
    config.length,
)

assert Z_R_test.shape == (
    len(DETECTORS),
    config.length,
)

assert np.all(
    np.isfinite(X_R_test)
)

assert np.all(
    np.isfinite(Z_R_test)
)

assert np.allclose(
    Z_R_test.mean(axis=1),
    0.0,
    atol=1e-5,
)

assert np.allclose(
    Z_R_test.std(axis=1),
    1.0,
    atol=1e-5,
)

print("Real-domain preprocessing contract passed.")

## 8. Gaussian empirical-PSD control for the same epoch

A Gaussian G1 control is built using exactly the same:

- intrinsic source;
- GPS epoch;
- detector geometry;
- empirical PSD;
- optimal network SNR;
- preprocessing;
- M10 normalization.

The only uncontrolled quantity is the stochastic noise realization.

Unlike M11.4-C, G1 and R cannot share an underlying Gaussian white draw because the real detector strain is an observed stochastic process rather than a generated realization.

Therefore single-example G1–R differences are diagnostic only and must not be interpreted as paired causal shifts.


In [ ]:
def generate_white_frequency_draw(
    flength,
    seed,
):
    """
    Generate the dimensionless Gaussian frequency-domain random draw
    used to construct a real time series.

    Interior positive-frequency bins contain independent complex
    standard-normal draws.

    DC and Nyquist bins are real.
    """
    rng = np.random.default_rng(seed)

    real = rng.normal(
        size=flength
    )

    imag = rng.normal(
        size=flength
    )

    z = real + 1j * imag

    # Real-valued time series constraints for rFFT endpoints.
    z[0] = real[0] + 0j
    z[-1] = real[-1] + 0j

    return z

In [ ]:
def color_frequency_draw_with_psd(
    z,
    psd,
):
    """
    Color one common dimensionless frequency draw with a
    one-sided PSD.

    Returns a PyCBC FrequencySeries.
    """
    s = np.asarray(
        psd.numpy(),
        dtype=np.float64,
    )

    if len(z) != len(s):
        raise ValueError(
            "White draw and PSD length mismatch."
        )

    df = float(psd.delta_f)

    out = np.zeros(
        len(s),
        dtype=np.complex128,
    )

    valid = (
        np.isfinite(s)
        & (s > 0)
    )

    scale = np.zeros_like(s)

    scale[valid] = np.sqrt(
        s[valid]
        / (4.0 * df)
    )

    out[valid] = (
        z[valid]
        * scale[valid]
    )

    # Endpoint bins must be purely real.
    out[0] = np.real(out[0]) + 0j
    out[-1] = np.real(out[-1]) + 0j

    return FrequencySeries(
        out,
        delta_f=df,
    )

In [ ]:
def build_G1_control_input(
    network,
    pair_seed,
):
    noises = {}

    context_start = (
        float(
            network.placement.segment_start_time
        )
        - config.processing_context_start_seconds
    )

    for k, ifo in enumerate(DETECTORS):

        detector_seed = (
            pair_seed
            + 10_000 * k
        )

        z = generate_white_frequency_draw(
            flength=len(
                empirical_psds_processing[
                    ifo
                ]
            ),
            seed=detector_seed,
        )

        nf = color_frequency_draw_with_psd(
            z,
            empirical_psds_processing[
                ifo
            ],
        )

        nt = nf.to_timeseries()

        nt = injector.set_strain_start_time(
            strain=nt,
            start_time=context_start,
            expected_length=config.processing_length,
        )

        noises[ifo] = nt

    injected = injector.inject_network(
        noises=noises,
        signals=network.projection.strains,
    )

    strains = {
        ifo: injected[ifo].strain
        for ifo in DETECTORS
    }

    processed = processor.process_network(
        strains=strains,
        psds=empirical_psds_processing,
    )

    X = np.stack(
        [
            np.asarray(
                processed[ifo],
                dtype=np.float64,
            )
            for ifo in DETECTORS
        ],
        axis=0,
    )

    Z, _, _ = m10_input_zscore(
        X,
        M10_INPUT_NORM_CONFIG,
    )

    return {
        "X": X,
        "Z": Z.astype(np.float32),
    }

In [ ]:
#First sanity

G1_test = build_G1_control_input(
    network=test_net_R["network"],
    pair_seed=200_000,
)

In [ ]:
rows = []

for k, ifo in enumerate(DETECTORS):

    rows.append({
        "ifo": ifo,

        "raw_std_G1":
            np.std(
                G1_test["X"][k]
            ),

        "raw_std_R":
            np.std(
                X_R_test[k]
            ),

        "raw_std_ratio_R_G1":
            np.std(
                X_R_test[k]
            )
            / np.std(
                G1_test["X"][k]
            ),

        "z_corr_R_G1":
            np.corrcoef(
                Z_R_test[k],
                G1_test["Z"][k],
            )[0, 1],

        "z_rmse_R_G1":
            np.sqrt(
                np.mean(
                    (
                        Z_R_test[k]
                        - G1_test["Z"][k]
                    )**2
                )
            ),
    })

df_M115_single = pd.DataFrame(rows)

df_M115_single

### Interpretation of the single-epoch comparison

The G1 and R examples do not share the same stochastic noise realization.

Therefore quantities such as

$$
\|Z_R-Z_{G1}\|
$$

or their direct time-series correlation contain both:

$$
\text{ordinary noise-realization variability}
$$

and

$$
\text{possible real-noise domain shift}.
$$

They are useful as technical sanity checks, but they are not the principal scientific metrics of M11.5.

The G1-versus-R question must therefore be answered at the ensemble-distribution level.

# M11.5-A — Single-epoch validation conclusion

The technical contract required to inject synthetic GWs into actual off-source detector strain is now established.

For the tested real-noise epoch:

- H1, L1 and V1 strain segments are available and finite;
- the processing-context window has the expected duration;
- the synthetic GW is projected using the physical GPS epoch of the real strain;
- the luminosity distance is adjusted using the empirical PSD to enforce

$$
\rho_{\rm net,opt}=15;
$$

- the full M11.1 projected waveform is injected on the absolute detector time axis;
- the resulting real-noise observation is processed with the same empirical PSD used to define its optimal SNR;
- the final output satisfies the M10 input contract

$$
X\in\mathbb R^{3\times16384};
$$

- the frozen M10 per-sample/per-detector z-score produces finite, zero-mean, unit-variance channels;
- a matched Gaussian G1 control can be generated at the same epoch, geometry, PSD and optimal SNR.

The single G1 and R realizations must not be interpreted as a paired stochastic comparison because their underlying noise realizations are different.

M11.5-A therefore validates the construction only.

The scientific G1-versus-R comparison will be performed at the ensemble level in M11.5-B.

# M11.5-B — Epoch-matched G1 versus real off-source noise

M11.5-A validated the technical construction required to inject a synthetic GW into actual off-source detector strain.

M11.5-B now performs the scientific comparison between

$$
G1 =
\text{Gaussian noise generated from the empirical PSD},
$$

and

$$
R =
\text{actual real off-source detector strain}.
$$

The experimental unit is the real detector epoch $j$.

For every selected GPS center,

$$
t_j,
$$

a synthetic BBH signal is projected using

$$
t_{\rm geo}=t_j,
$$

and its luminosity distance is adjusted such that

$$
\rho_{\rm net,opt}=15.
$$

The real-domain observation is

$$
R_j=h_j+n^{\rm real}_j.
$$

For the same epoch, source, geometry, PSD and optimal SNR, several Gaussian controls are generated:

$$
G1_{j,k}
=
h_j+n^{\rm Gaussian}_{j,k}.
$$

The comparison is therefore epoch matched but not stochastic-realization matched.

The objectives are to compare G1 and R at four levels:

1. processed-strain statistics;
2. realized matched-filter SNR;
3. frozen-M10 latent representation;
4. frozen-M10 parameter predictions.

A systematic difference between G1 and R would indicate detector-noise structure that is not captured by a stationary Gaussian process described only by the empirical PSD.

In [ ]:
N_R = min(
    30,
    len(valid_centers_R),
)

N_G1_PER_EPOCH = 3

M115_B_SEED0 = 300_000

centers_B = np.asarray(
    valid_centers_R[:N_R],
    dtype=float,
)

print("Real epochs:", N_R)
print("G1 controls per epoch:", N_G1_PER_EPOCH)
print(
    "Total G1 samples:",
    N_R * N_G1_PER_EPOCH,
)

In [ ]:
def extract_output_segment_from_context(
    strain_context,
    network,
):
    output_start = float(
        network.placement.segment_start_time
    )

    output_end = (
        output_start
        + config.duration
    )

    segment = strain_context.time_slice(
        output_start,
        output_end,
    )

    if len(segment) != config.length:
        raise ValueError(
            "Unexpected output segment length: "
            f"{len(segment)} != {config.length}"
        )

    return segment

## Realized-SNR diagnostic

For each injected observation we compute the fixed-template, signed realized network SNR

$$
\rho_{\rm net,real}
=
\frac{
\sum_I(d_I|h_I)
}{
\sqrt{
\sum_I(h_I|h_I)
}
}.
$$

The corresponding fluctuation is

$$
\Delta\rho
=
\rho_{\rm net,real}
-
\rho_{\rm net,opt}.
$$

For stationary Gaussian G1 noise generated consistently from the same PSD,

$$
\Delta\rho
\sim
\mathcal N(0,1)
$$

is expected.

The R distribution is not forced to satisfy this result and therefore provides a diagnostic of real detector-noise behavior beyond the Gaussian PSD model.

In [ ]:
def noise_weighted_inner_product(
    a,
    b,
    psd,
    f_low,
):
    af = a.to_frequencyseries()
    bf = b.to_frequencyseries()

    f = af.sample_frequencies.numpy()

    aa = af.numpy()
    bb = bf.numpy()
    ss = psd.numpy()

    mask = (
        (f >= f_low)
        & (ss > 0)
        & np.isfinite(ss)
        & np.isfinite(aa)
        & np.isfinite(bb)
    )

    df = float(af.delta_f)

    value = (
        4.0
        * df
        * np.real(
            np.sum(
                np.conjugate(aa[mask])
                * bb[mask]
                / ss[mask]
            )
        )
    )

    return float(value)

In [ ]:
def compute_signed_network_realized_snr(
    data_segments,
    signal_segments,
    psds,
):
    dh_sum = 0.0
    hh_sum = 0.0

    detector_rows = {}

    for ifo in DETECTORS:

        dh = noise_weighted_inner_product(
            data_segments[ifo],
            signal_segments[ifo],
            psds[ifo],
            config.low_frequency_cutoff,
        )

        hh = noise_weighted_inner_product(
            signal_segments[ifo],
            signal_segments[ifo],
            psds[ifo],
            config.low_frequency_cutoff,
        )

        rho_opt_ifo = np.sqrt(hh)
        rho_real_ifo = dh / rho_opt_ifo

        detector_rows[ifo] = {
            "rho_opt": rho_opt_ifo,
            "rho_real": rho_real_ifo,
            "delta_rho":
                rho_real_ifo
                - rho_opt_ifo,
        }

        dh_sum += dh
        hh_sum += hh

    rho_opt_network = np.sqrt(
        hh_sum
    )

    rho_real_network = (
        dh_sum
        / rho_opt_network
    )

    return {
        "detectors": detector_rows,
        "rho_opt_network":
            rho_opt_network,
        "rho_real_network":
            rho_real_network,
        "delta_rho_network":
            (
                rho_real_network
                - rho_opt_network
            ),
    }

In [ ]:
def build_R_sample(
    center_time,
):
    net_info = (
        build_network_at_real_epoch(
            center_time=center_time,
            target_rho=TARGET_RHO_R,
        )
    )

    network = net_info["network"]

    noises_R = (
        extract_real_processing_noise(
            raw_strains_long,
            network,
        )
    )

    injected_R = (
        injector.inject_network(
            noises=noises_R,
            signals=network.projection.strains,
        )
    )

    strains_context_R = {
        ifo:
            injected_R[ifo].strain
        for ifo in DETECTORS
    }

    data_4s_R = {
        ifo:
            extract_output_segment_from_context(
                strains_context_R[ifo],
                network,
            )
        for ifo in DETECTORS
    }

    processed_R = (
        processor.process_network(
            strains=strains_context_R,
            psds=empirical_psds_processing,
        )
    )

    X_R = np.stack(
        [
            np.asarray(
                processed_R[ifo],
                dtype=np.float64,
            )
            for ifo in DETECTORS
        ],
        axis=0,
    )

    Z_R, _, _ = (
        m10_input_zscore(
            X_R,
            M10_INPUT_NORM_CONFIG,
        )
    )

    snr_R = (
        compute_signed_network_realized_snr(
            data_segments=data_4s_R,
            signal_segments=network.signal_segments,
            psds=empirical_psds_4s,
        )
    )

    return {
        "center_time":
            float(center_time),

        "network":
            network,

        "distance":
            float(
                net_info["distance"]
            ),

        "rho_opt":
            float(
                net_info["network_snr"]
            ),

        "detector_snrs":
            net_info[
                "detector_snrs"
            ],

        "X":
            X_R,

        "Z":
            Z_R.astype(
                np.float32
            ),

        "realized_snr":
            snr_R,

        "injection":
            injected_R,
    }

In [ ]:
def build_G1_sample(
    network,
    gaussian_seed,
):
    context_start = (
        float(
            network
            .placement
            .segment_start_time
        )
        - config.processing_context_start_seconds
    )

    noises = {}

    for k, ifo in enumerate(
        DETECTORS
    ):
        detector_seed = (
            gaussian_seed
            + 10_000 * k
        )

        z = (
            generate_white_frequency_draw(
                flength=len(
                    empirical_psds_processing[
                        ifo
                    ]
                ),
                seed=detector_seed,
            )
        )

        nf = (
            color_frequency_draw_with_psd(
                z,
                empirical_psds_processing[
                    ifo
                ],
            )
        )

        nt = nf.to_timeseries()

        nt = (
            injector.set_strain_start_time(
                strain=nt,
                start_time=context_start,
                expected_length=(
                    config.processing_length
                ),
            )
        )

        noises[ifo] = nt

    injected = (
        injector.inject_network(
            noises=noises,
            signals=(
                network
                .projection
                .strains
            ),
        )
    )

    strains_context = {
        ifo:
            injected[ifo].strain
        for ifo in DETECTORS
    }

    data_4s = {
        ifo:
            extract_output_segment_from_context(
                strains_context[ifo],
                network,
            )
        for ifo in DETECTORS
    }

    processed = (
        processor.process_network(
            strains=strains_context,
            psds=empirical_psds_processing,
        )
    )

    X = np.stack(
        [
            np.asarray(
                processed[ifo],
                dtype=np.float64,
            )
            for ifo in DETECTORS
        ],
        axis=0,
    )

    Z, _, _ = (
        m10_input_zscore(
            X,
            M10_INPUT_NORM_CONFIG,
        )
    )

    snr = (
        compute_signed_network_realized_snr(
            data_segments=data_4s,
            signal_segments=network.signal_segments,
            psds=empirical_psds_4s,
        )
    )

    return {
        "X":
            X,

        "Z":
            Z.astype(
                np.float32
            ),

        "realized_snr":
            snr,

        "injection":
            injected,
    }

## Epoch-matched ensemble construction

For every real epoch $j$:

1. one real-noise observation $R_j$ is constructed;
2. the same epoch-specific signal network is reused;
3. three independent G1 Gaussian controls are generated.

Thus all G1 controls associated with epoch $j$ share the same

$$
h_j,
\quad
t_j,
\quad
S_n(f),
\quad
\rho_{\rm net,opt}=15,
$$

as the corresponding real observation.

In [ ]:
R_samples_B = []
G1_samples_B = []

for j, center_time in enumerate(
    centers_B
):
    r_sample = build_R_sample(
        center_time
    )

    R_samples_B.append(
        {
            "epoch_index": j,
            **r_sample,
        }
    )

    for k in range(
        N_G1_PER_EPOCH
    ):
        seed = (
            M115_B_SEED0
            + 1000 * j
            + k
        )

        g_sample = (
            build_G1_sample(
                network=r_sample[
                    "network"
                ],
                gaussian_seed=seed,
            )
        )

        G1_samples_B.append(
            {
                "epoch_index": j,
                "g1_index": k,
                "gaussian_seed": seed,
                "center_time":
                    float(
                        center_time
                    ),
                **g_sample,
            }
        )

    if (j + 1) % 5 == 0:
        print(
            f"{j + 1}/{N_R} epochs built"
        )

In [ ]:
assert len(R_samples_B) == N_R

assert (
    len(G1_samples_B)
    == N_R * N_G1_PER_EPOCH
)

for sample in R_samples_B:
    assert sample["Z"].shape == (
        len(DETECTORS),
        config.length,
    )

    assert np.all(
        np.isfinite(
            sample["Z"]
        )
    )

for sample in G1_samples_B:
    assert sample["Z"].shape == (
        len(DETECTORS),
        config.length,
    )

    assert np.all(
        np.isfinite(
            sample["Z"]
        )
    )

print(
    "M11.5-B ensemble contracts passed."
)

In [ ]:
rows = []

for r in R_samples_B:

    row = {
        "epoch_index":
            r["epoch_index"],

        "center_time":
            r["center_time"],

        "distance_mpc":
            r["distance"],

        "network_rho_opt":
            r["rho_opt"],

        "network_rho_real":
            r[
                "realized_snr"
            ][
                "rho_real_network"
            ],

        "network_delta_rho":
            r[
                "realized_snr"
            ][
                "delta_rho_network"
            ],
    }

    for ifo in DETECTORS:
        row[
            f"{ifo}_rho_opt"
        ] = (
            r["realized_snr"]
            ["detectors"]
            [ifo]
            ["rho_opt"]
        )

        row[
            f"{ifo}_rho_real"
        ] = (
            r["realized_snr"]
            ["detectors"]
            [ifo]
            ["rho_real"]
        )

    rows.append(row)

df_R_metadata_B = pd.DataFrame(
    rows
)

df_R_metadata_B

In [ ]:
rows = []

for g in G1_samples_B:

    rows.append({
        "epoch_index":
            g["epoch_index"],

        "g1_index":
            g["g1_index"],

        "center_time":
            g["center_time"],

        "network_rho_opt":
            g[
                "realized_snr"
            ][
                "rho_opt_network"
            ],

        "network_rho_real":
            g[
                "realized_snr"
            ][
                "rho_real_network"
            ],

        "network_delta_rho":
            g[
                "realized_snr"
            ][
                "delta_rho_network"
            ],
    })

df_G1_metadata_B = pd.DataFrame(
    rows
)

df_G1_metadata_B

In [ ]:
print(
    "R results:\n",
    df_R_metadata_B[
        "network_rho_opt"
    ].describe()
)

print(
    "G1 results:\n",
    df_G1_metadata_B[
        "network_rho_opt"
    ].describe()
)

In [ ]:
def summarize_delta_rho(
    x,
):
    x = np.asarray(x)

    return {
        "n":
            len(x),

        "mean":
            np.mean(x),

        "std":
            np.std(
                x,
                ddof=1,
            ),

        "q05":
            np.quantile(
                x,
                0.05,
            ),

        "q50":
            np.quantile(
                x,
                0.50,
            ),

        "q95":
            np.quantile(
                x,
                0.95,
            ),

        "frac_abs_gt_2":
            np.mean(
                np.abs(x) > 2
            ),

        "frac_abs_gt_3":
            np.mean(
                np.abs(x) > 3
            ),
    }

In [ ]:
delta_R_B = (
    df_R_metadata_B[
        "network_delta_rho"
    ].to_numpy()
)

delta_G1_B = (
    df_G1_metadata_B[
        "network_delta_rho"
    ].to_numpy()
)

df_delta_rho_summary_B = (
    pd.DataFrame(
        [
            {
                "domain": "G1",
                **summarize_delta_rho(
                    delta_G1_B
                ),
            },
            {
                "domain": "R",
                **summarize_delta_rho(
                    delta_R_B
                ),
            },
        ]
    )
)

df_delta_rho_summary_B

In [ ]:
plt.figure(
    figsize=(8, 4)
)

plt.hist(
    delta_G1_B,
    bins=15,
    density=True,
    alpha=0.5,
    label="G1",
)

plt.hist(
    delta_R_B,
    bins=15,
    density=True,
    alpha=0.5,
    label="R",
)

plt.xlabel(
    r"$\Delta\rho$"
)

plt.ylabel(
    "Density"
)

plt.title(
    "Realized-SNR fluctuation: G1 vs R"
)

plt.legend()
plt.show()

In [ ]:
ks_delta_B = ks_2samp(
    delta_G1_B,
    delta_R_B,
)

wasserstein_delta_B = (
    wasserstein_distance(
        delta_G1_B,
        delta_R_B,
    )
)

print(
    "KS statistic:",
    ks_delta_B.statistic,
)

print(
    "KS p-value:",
    ks_delta_B.pvalue,
)

print(
    "Wasserstein distance:",
    wasserstein_delta_B,
)

- KS pregunta si las CDFs difieren;
- Wasserstein mide cuánto habría que desplazar la masa de probabilidad para convertir una distribución en la otra.

In [ ]:
rows = []

for r in R_samples_B:

    for k, ifo in enumerate(
        DETECTORS
    ):
        x = r["X"][k]

        rows.append({
            "domain": "R",
            "epoch_index":
                r["epoch_index"],
            "ifo": ifo,
            "std":
                np.std(x),
            "mean":
                np.mean(x),
            "abs_q99":
                np.quantile(
                    np.abs(x),
                    0.99,
                ),
            "abs_max":
                np.max(
                    np.abs(x)
                ),
        })

for g in G1_samples_B:

    for k, ifo in enumerate(
        DETECTORS
    ):
        x = g["X"][k]

        rows.append({
            "domain": "G1",
            "epoch_index":
                g["epoch_index"],
            "ifo": ifo,
            "std":
                np.std(x),
            "mean":
                np.mean(x),
            "abs_q99":
                np.quantile(
                    np.abs(x),
                    0.99,
                ),
            "abs_max":
                np.max(
                    np.abs(x)
                ),
        })

df_processed_stats_B = pd.DataFrame(
    rows
)

In [ ]:
df_processed_summary_B = (
    df_processed_stats_B
    .groupby(
        [
            "domain",
            "ifo",
        ]
    )
    .agg(
        mean_std=(
            "std",
            "mean",
        ),
        std_std=(
            "std",
            "std",
        ),
        median_q99=(
            "abs_q99",
            "median",
        ),
        q90_q99=(
            "abs_q99",
            lambda x:
                np.quantile(
                    x,
                    0.90,
                ),
        ),
        median_abs_max=(
            "abs_max",
            "median",
        ),
    )
    .reset_index()
)

df_processed_summary_B

In [ ]:
rows = []

for ifo in DETECTORS:

    for metric in [
        "std",
        "abs_q99",
        "abs_max",
    ]:

        a = (
            df_processed_stats_B
            .query(
                "domain == 'G1' "
                "and ifo == @ifo"
            )[metric]
            .to_numpy()
        )

        b = (
            df_processed_stats_B
            .query(
                "domain == 'R' "
                "and ifo == @ifo"
            )[metric]
            .to_numpy()
        )

        ks = ks_2samp(
            a,
            b,
        )

        rows.append({
            "ifo":
                ifo,

            "metric":
                metric,

            "mean_G1":
                np.mean(a),

            "mean_R":
                np.mean(b),

            "KS":
                ks.statistic,

            "KS_p":
                ks.pvalue,

            "Wasserstein":
                wasserstein_distance(
                    a,
                    b,
                ),
        })

df_processed_tests_B = pd.DataFrame(
    rows
)

df_processed_tests_B

In [ ]:
Z_R_B = np.stack(
    [
        r["Z"]
        for r in R_samples_B
    ],
    axis=0,
)

Z_G1_B = np.stack(
    [
        g["Z"]
        for g in G1_samples_B
    ],
    axis=0,
)

print(
    Z_R_B.shape,
    Z_G1_B.shape,
)

## Load M10 model

In [ ]:
from pathlib import Path
import importlib
import torch

DATA_ROOT = Path("/data/vserrano/cbc_pe_data")
DATASET_ID = "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"

PRED_PATH = (
    DATA_ROOT
    / "results"
    / DATASET_ID
    / "m10_inputzscore_500k_cal_test_predictions_embeddings.npz"
)

CHECKPOINT_PATH = (
    DATA_ROOT
    / "models"
    / "checkpoints"
    / DATASET_ID
    / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_"
        "SimpleCNN_ResidualDilated_"
        "M10_inputzscore_resdilated_emb64_d124_train400k_"
        "MSELoss_seed123_checkpoint.pt"
    )
)

assert PRED_PATH.exists(), PRED_PATH
assert CHECKPOINT_PATH.exists(), CHECKPOINT_PATH

print("Prediction artifact:", PRED_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

In [ ]:
pred_data = np.load(
    PRED_PATH,
    allow_pickle=True,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
)

# Frozen M10 metadata
label_mean = pred_data["y_mean"].astype(np.float32)
label_std = pred_data["y_std"].astype(np.float32)
label_names = [
    str(x)
    for x in pred_data["label_names"].tolist()
]

model_config_pred = pred_data["model_config"].item()
input_norm_config = pred_data["input_normalization"].item()

model_config_ckpt = checkpoint["model_config"]

print("Device:", device)
print("Labels:", label_names)
print("Input normalization:", input_norm_config)
print("Checkpoint epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["best_val_loss"])

In [ ]:
assert model_config_ckpt == model_config_pred

assert np.allclose(
    checkpoint["y_mean"],
    label_mean,
)

assert np.allclose(
    checkpoint["y_std"],
    label_std,
)

assert (
    model_config_ckpt["input_normalization"]
    == input_norm_config
)

assert input_norm_config == {
    "enabled": True,
    "mode": "per_sample_per_detector_zscore",
    "eps": 1e-6,
}

print("M10 model contract verified.")

In [ ]:
model_config = model_config_ckpt

network_module = importlib.import_module(
    "src.models.network"
)

class_name = model_config["class_name"]
model_kwargs = dict(
    model_config["model_kwargs"]
)

model_class = getattr(
    network_module,
    class_name,
)

model = model_class(
    **model_kwargs,
).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print(
    f"Loaded frozen M10 model: {class_name}"
)
print("Model kwargs:")
model_kwargs

Once loaded, we can do the inference

In [ ]:
model.eval()

with torch.no_grad():

    pred_std_R_B, emb_R_B = model(
        torch.as_tensor(
            Z_R_B,
            dtype=torch.float32,
            device=device,
        ),
        return_embedding=True,
    )

    pred_std_G1_B, emb_G1_B = model(
        torch.as_tensor(
            Z_G1_B,
            dtype=torch.float32,
            device=device,
        ),
        return_embedding=True,
    )

pred_std_R_B = (
    pred_std_R_B
    .cpu()
    .numpy()
)

pred_std_G1_B = (
    pred_std_G1_B
    .cpu()
    .numpy()
)

emb_R_B = (
    emb_R_B
    .cpu()
    .numpy()
)

emb_G1_B = (
    emb_G1_B
    .cpu()
    .numpy()
)

In [ ]:
pred_phys_R_B = (
    pred_std_R_B
    * label_std[None, :]
    + label_mean[None, :]
)

pred_phys_G1_B = (
    pred_std_G1_B
    * label_std[None, :]
    + label_mean[None, :]
)

In [ ]:
LABELS = ["chirp_mass", "total_mass", "chi_eff"]

true_B = np.asarray(
    [
        BASE_PARAMS_R.chirp_mass,
        BASE_PARAMS_R.total_mass,
        BASE_PARAMS_R.chi_eff,
    ],
    dtype=float,
)

print(
    dict(
        zip(
            LABELS,
            true_B,
        )
    )
)

G1 vs R performance

In [ ]:
rows = []

for j, label in enumerate(
    LABELS
):

    truth = true_B[j]

    for domain, pred in [
        (
            "G1",
            pred_phys_G1_B,
        ),
        (
            "R",
            pred_phys_R_B,
        ),
    ]:

        residual = (
            pred[:, j]
            - truth
        )

        rows.append({
            "domain":
                domain,

            "label":
                label,

            "mean_prediction":
                np.mean(
                    pred[:, j]
                ),

            "bias":
                np.mean(
                    residual
                ),

            "prediction_std":
                np.std(
                    pred[:, j],
                    ddof=1,
                ),

            "rmse":
                np.sqrt(
                    np.mean(
                        residual**2
                    )
                ),

            "mae":
                np.mean(
                    np.abs(
                        residual
                    )
                ),
        })

df_prediction_performance_B = (
    pd.DataFrame(rows)
)

df_prediction_performance_B

In [ ]:
rows = []

for j, label in enumerate(
    LABELS
):

    pG = (
        pred_phys_G1_B[:, j]
    )

    pR = (
        pred_phys_R_B[:, j]
    )

    ks = ks_2samp(
        pG,
        pR,
    )

    rows.append({
        "label":
            label,

        "KS":
            ks.statistic,

        "KS_p":
            ks.pvalue,

        "Wasserstein":
            wasserstein_distance(
                pG,
                pR,
            ),

        "mean_shift_R_minus_G1":
            (
                np.mean(pR)
                - np.mean(pG)
            ),
    })

df_prediction_distribution_B = (
    pd.DataFrame(rows)
)

df_prediction_distribution_B

In [ ]:
rows = []

for epoch_index in range(
    N_R
):

    mask_g = np.asarray(
        [
            g["epoch_index"]
            == epoch_index
            for g in G1_samples_B
        ]
    )

    pred_g_epoch = (
        pred_phys_G1_B[
            mask_g
        ]
    )

    pred_r_epoch = (
        pred_phys_R_B[
            epoch_index
        ]
    )

    for j, label in enumerate(
        LABELS
    ):

        mu_g = np.mean(
            pred_g_epoch[:, j]
        )

        sigma_g = np.std(
            pred_g_epoch[:, j],
            ddof=1,
        )

        delta = (
            pred_r_epoch[j]
            - mu_g
        )

        z_epoch = (
            delta / sigma_g
            if sigma_g > 0
            else np.nan
        )

        rows.append({
            "epoch_index":
                epoch_index,

            "center_time":
                centers_B[
                    epoch_index
                ],

            "label":
                label,

            "R_prediction":
                pred_r_epoch[j],

            "G1_epoch_mean":
                mu_g,

            "G1_epoch_std":
                sigma_g,

            "R_minus_G1_mean":
                delta,

            "epoch_standardized_shift":
                z_epoch,
        })

df_epoch_matched_prediction_B = (
    pd.DataFrame(rows)
)

df_epoch_matched_prediction_B

In [ ]:
df_epoch_matched_summary_B = (
    df_epoch_matched_prediction_B
    .groupby(
        "label"
    )
    .agg(
        mean_shift=(
            "R_minus_G1_mean",
            "mean",
        ),

        std_shift=(
            "R_minus_G1_mean",
            "std",
        ),

        median_abs_shift=(
            "R_minus_G1_mean",
            lambda x:
                np.median(
                    np.abs(x)
                ),
        ),

        q90_abs_shift=(
            "R_minus_G1_mean",
            lambda x:
                np.quantile(
                    np.abs(x),
                    0.90,
                ),
        ),

        median_abs_epoch_z=(
            "epoch_standardized_shift",
            lambda x:
                np.nanmedian(
                    np.abs(x)
                ),
        ),
    )
    .reset_index()
)

df_epoch_matched_summary_B

In [ ]:
centroid_G1_B = np.mean(
    emb_G1_B,
    axis=0,
)

centroid_R_B = np.mean(
    emb_R_B,
    axis=0,
)

In [ ]:
centroid_l2_B = np.linalg.norm(
    centroid_R_B
    - centroid_G1_B
)

within_G1_B = np.linalg.norm(
    emb_G1_B
    - centroid_G1_B,
    axis=1,
)

within_R_B = np.linalg.norm(
    emb_R_B
    - centroid_R_B,
    axis=1,
)

print(
    "Centroid G1-R distance:",
    centroid_l2_B,
)

print(
    "Mean within-G1 radius:",
    within_G1_B.mean(),
)

print(
    "Mean within-R radius:",
    within_R_B.mean(),
)

print(
    "Centroid shift / G1 radius:",
    centroid_l2_B
    / within_G1_B.mean(),
)

In [ ]:
rows = []

for epoch_index in range(
    N_R
):

    mask_g = np.asarray(
        [
            g["epoch_index"]
            == epoch_index
            for g in G1_samples_B
        ]
    )

    emb_g_epoch = (
        emb_G1_B[
            mask_g
        ]
    )

    emb_r = (
        emb_R_B[
            epoch_index
        ]
    )

    mu_g = np.mean(
        emb_g_epoch,
        axis=0,
    )

    dist_R_to_G1_center = (
        np.linalg.norm(
            emb_r - mu_g
        )
    )

    g_internal_dist = (
        np.linalg.norm(
            emb_g_epoch
            - mu_g,
            axis=1,
        )
    )

    rows.append({
        "epoch_index":
            epoch_index,

        "center_time":
            centers_B[
                epoch_index
            ],

        "R_to_G1_epoch_centroid":
            dist_R_to_G1_center,

        "mean_G1_internal_radius":
            np.mean(
                g_internal_dist
            ),

        "relative_epoch_embedding_shift":
            (
                dist_R_to_G1_center
                / np.mean(
                    g_internal_dist
                )
                if np.mean(
                    g_internal_dist
                ) > 0
                else np.nan
            ),
    })

df_epoch_embedding_B = pd.DataFrame(
    rows
)

df_epoch_embedding_B.describe()

## $\Delta\rho$ dependency with CNN error

In [ ]:
residual_R_B = (
    pred_phys_R_B
    - true_B[None, :]
)

In [ ]:
from scipy.stats import (
    pearsonr,
    spearmanr,
)

rows = []

for j, label in enumerate(
    LABELS
):

    pearson = pearsonr(
        delta_R_B,
        residual_R_B[:, j],
    )

    spearman = spearmanr(
        delta_R_B,
        residual_R_B[:, j],
    )

    rows.append({
        "label":
            label,

        "pearson_r":
            pearson.statistic,

        "pearson_p":
            pearson.pvalue,

        "spearman_rho":
            spearman.statistic,

        "spearman_p":
            spearman.pvalue,
    })

df_R_snr_residual_corr_B = (
    pd.DataFrame(rows)
)

df_R_snr_residual_corr_B

In [ ]:
df_epoch_diagnostic_B = (
    df_R_metadata_B[
        [
            "epoch_index",
            "center_time",
            "network_delta_rho",
        ]
    ]
    .merge(
        df_epoch_embedding_B,
        on=[
            "epoch_index",
            "center_time",
        ],
        how="left",
    )
)

In [ ]:
residual_scale_B = (
    np.std(
        pred_phys_G1_B
        - true_B[None, :],
        axis=0,
        ddof=1,
    )
)

standardized_residual_R_B = (
    residual_R_B
    / residual_scale_B[
        None,
        :
    ]
)

df_epoch_diagnostic_B[
    "cnn_residual_norm"
] = np.linalg.norm(
    standardized_residual_R_B,
    axis=1,
)

df_epoch_diagnostic_B = (
    df_epoch_diagnostic_B
    .sort_values(
        "cnn_residual_norm",
        ascending=False,
    )
)

df_epoch_diagnostic_B.head(10)

In [ ]:
for j, label in enumerate(
    LABELS
):

    plt.figure(
        figsize=(7, 4)
    )

    plt.hist(
        pred_phys_G1_B[:, j],
        bins=15,
        alpha=0.5,
        density=True,
        label="G1",
    )

    plt.hist(
        pred_phys_R_B[:, j],
        bins=15,
        alpha=0.5,
        density=True,
        label="R",
    )

    plt.axvline(
        true_B[j],
        linestyle="--",
        label="Truth",
    )

    plt.xlabel(label)
    plt.ylabel("Density")

    plt.title(
        f"{label}: G1 vs real-noise predictions"
    )

    plt.legend()
    plt.show()

In [ ]:
for label in LABELS:

    d = (
        df_epoch_matched_prediction_B
        .query(
            "label == @label"
        )
    )

    plt.figure(
        figsize=(9, 4)
    )

    plt.plot(
        d["epoch_index"],
        d["R_minus_G1_mean"],
        marker="o",
    )

    plt.axhline(
        0.0,
        linestyle="--",
    )

    plt.xlabel(
        "Epoch index"
    )

    plt.ylabel(
        "R - mean(G1)"
    )

    plt.title(
        f"{label}: epoch-matched domain shift"
    )

    plt.show()

In [ ]:
plt.figure(
    figsize=(9, 4)
)

plt.plot(
    df_epoch_embedding_B[
        "epoch_index"
    ],
    df_epoch_embedding_B[
        "relative_epoch_embedding_shift"
    ],
    marker="o",
)

plt.axhline(
    1.0,
    linestyle="--",
)

plt.xlabel(
    "Epoch index"
)

plt.ylabel(
    "R-to-G1 distance / G1 internal radius"
)

plt.title(
    "Epoch-matched latent-domain displacement"
)

plt.show()

## Interpretation criteria

M11.5-B is not designed around one binary hypothesis test.

Evidence for a meaningful G1-to-R domain shift will be assessed jointly from:

- realized-SNR distribution changes;
- processed-strain tail statistics;
- latent-space displacement;
- prediction bias and dispersion;
- epoch-matched G1-versus-R shifts;
- concentration of effects in a small number of real epochs.

A small KS p-value alone will not be treated as sufficient evidence of a scientifically important domain shift.

Conversely, failure to reject equality with the small current sample does not demonstrate that the domains are equivalent.

The emphasis is on effect sizes, physical consistency and agreement across diagnostics.

# M11.5-B — Results and interpretation

M11.5-B compares the frozen M10 predictor under two noise domains:

$$
G1 =
\text{Gaussian noise generated from an empirical detector PSD},
$$

and

$$
R =
\text{actual real off-source detector strain}.
$$

For every real detector epoch $j$, the comparison keeps fixed:

- the intrinsic BBH parameters;
- the GPS epoch;
- the detector geometry and antenna response;
- the empirical PSD;
- the preprocessing pipeline;
- the M10 per-sample/per-detector z-score;
- the target network optimal SNR,

$$
\rho_{\rm net,opt}=15.
$$

The real observation is

$$
R_j=h_j+n^{\rm real}_j,
$$

while three Gaussian controls are generated for the same epoch,

$$
G1_{j,k}
=
h_j+n^{\rm Gaussian}_{j,k}.
$$

The experiment therefore tests whether an empirical PSD is sufficient to reproduce the statistical domain encountered by the frozen M10 network when actual detector strain is used.

---

## 1. Important caveat: G1 realized-SNR diagnostic failed its sanity check

The realized-SNR statistic was intended to compare

$$
\Delta\rho
=
\rho_{\rm real}-\rho_{\rm opt}
$$

between G1 and R.

For the real-noise domain, the measured distribution was

$$
\overline{\Delta\rho}_R
\simeq
-0.52,
$$

with

$$
\sigma(\Delta\rho_R)
\simeq
1.02.
$$

The 5th and 95th percentiles were approximately

$$
q_{0.05}
\simeq
-2.34,
$$

$$
q_{0.95}
\simeq
0.96,
$$

with

$$
P(|\Delta\rho|>2)
\simeq
0.13,
$$

and no realization satisfying

$$
|\Delta\rho|>3.
$$

However, the G1 control produced

$$
\overline{\Delta\rho}_{G1}
\simeq
-4.85,
$$

and

$$
\sigma(\Delta\rho_{G1})
\simeq
47.6,
$$

which is incompatible with the previously validated Gaussian expectation

$$
\Delta\rho_{G1}
\sim
\mathcal N(0,1).
$$

Therefore the current G1 realized-SNR result is considered a **failed technical sanity check** rather than a physical result.

The corresponding G1-versus-R KS and Wasserstein statistics for $\Delta\rho$ must not be interpreted scientifically until this inconsistency is resolved.

The likely source is the mismatch between generating a longer processing-context Gaussian realization using the processing-resolution PSD and evaluating its cropped 4 s segment with the independently constructed 4 s PSD.

The realized-SNR comparison is therefore left pending for a dedicated validation step.

---

## 2. Processed-strain statistics reveal a strong detector-dependent G1-to-R shift

The strongest direct domain difference appears before the final M10 z-score.

The mean standard deviation of the processed strain is approximately

| Detector | G1 | R |
|---|---:|---:|
| H1 | 21.90 | 22.69 |
| L1 | 22.26 | 95.90 |
| V1 | 22.56 | 143.07 |

Thus H1 remains close to the Gaussian empirical-PSD control,

$$
\frac{\sigma_R}{\sigma_{G1}}
\simeq
1.04,
$$

whereas L1 and V1 exhibit much larger processed amplitudes,

$$
\frac{\sigma_R}{\sigma_{G1}}
\simeq
4.3
\qquad
{\rm L1},
$$

and

$$
\frac{\sigma_R}{\sigma_{G1}}
\simeq
6.3
\qquad
{\rm V1}.
$$

The discrepancy is also present in tail-sensitive statistics.

For the 99th percentile of the absolute processed strain,

$$
q_{0.99}(|X|),
$$

the median values are approximately

$$
56.6 \rightarrow 58.4
\qquad
{\rm H1},
$$

$$
58.1 \rightarrow 175.4
\qquad
{\rm L1},
$$

and

$$
58.1 \rightarrow 289.0
\qquad
{\rm V1},
$$

when moving from G1 to R.

The maximum absolute amplitudes show the same pattern.

For L1,

$$
|X|_{\max}
:
103
\rightarrow
226,
$$

and for V1,

$$
|X|_{\max}
:
91
\rightarrow
376.
$$

The distributional tests are correspondingly strong.

For L1 and V1, the KS statistic reaches

$$
KS=1
$$

for processed standard deviation, $q_{0.99}(|X|)$ and maximum absolute amplitude, meaning that the sampled G1 and R distributions are completely separated according to these statistics.

H1 shows a much smaller shift.

### Interpretation

The empirical PSD alone is clearly insufficient to reproduce the processed real-strain distribution observed in L1 and V1.

However, this result should not yet be identified uniquely with non-Gaussianity.

Possible contributions include:

- residual PSD mismatch between the PSD-estimation window and the tested real-noise region;
- non-stationarity;
- persistent instrumental spectral structure;
- transient disturbances;
- departures from Gaussianity;
- differences in the effective whitening normalization.

The magnitude and persistence of the L1/V1 effect justify a dedicated residual-noise characterization step.

---

## 3. M10 z-score removes the large scale mismatch but does not eliminate the domain shift

M10 applies a per-sample/per-detector normalization,

$$
Z_I(t)
=
\frac{
X_I(t)-\mu_I
}{
\sigma_I
}.
$$

Therefore the very large differences in processed scale between G1 and R are explicitly removed before the CNN receives the input.

This is an important result in itself.

The M10 z-score is highly effective at suppressing detector-dependent amplitude-scale mismatch.

However, the following latent-space and prediction results show that the real-noise domain remains substantially different even after this normalization.

Therefore the G1-to-R difference cannot be described solely as a channel-scale problem.

---

## 4. Real noise produces a strong latent-domain displacement

The 64-dimensional frozen-M10 embeddings show a clear separation between G1 and R.

The distance between the two global centroids is

$$
\|\mu_R-\mu_{G1}\|_2
\simeq
1.39.
$$

For comparison, the mean internal radius of the Gaussian G1 distribution is only

$$
\left\langle
\|
\mathbf z_{G1}-\mu_{G1}
\|_2
\right\rangle
\simeq
0.355.
$$

Therefore

$$
\frac{
\|\mu_R-\mu_{G1}\|_2
}{
\left\langle
\|
\mathbf z_{G1}-\mu_{G1}
\|_2
\right\rangle
}
\simeq
3.93.
$$

The G1-to-R centroid displacement is therefore almost four times the characteristic internal radius of the Gaussian domain.

The real-noise domain is also substantially broader:

$$
\left\langle
\|
\mathbf z_R-\mu_R
\|_2
\right\rangle
\simeq
0.784,
$$

compared with

$$
0.355
$$

for G1.

Thus R is both:

1. shifted relative to G1;
2. more dispersed internally.

This provides direct evidence that the frozen CNN represents real off-source strain differently from PSD-matched Gaussian noise, even after whitening and M10 z-score normalization.

---

## 5. Epoch-matched latent analysis confirms that the effect is systematic

The comparison is stronger when performed separately for each real epoch.

For every epoch $j$, the real embedding is compared with the centroid of its three G1 controls.

The relative displacement is defined as

$$
D_j
=
\frac{
\|
\mathbf z_{R,j}
-
\mu_{G1,j}
\|_2
}{
\left\langle
\|
\mathbf z_{G1,j,k}
-
\mu_{G1,j}
\|_2
\right\rangle_k
}.
$$

Across the 30 epochs,

$$
\langle D\rangle
\simeq
5.62,
$$

with median

$$
D_{\rm med}
\simeq
4.64.
$$

The observed range is approximately

$$
2.0
\lesssim
D
\lesssim
11.6.
$$

Therefore even the most G1-like real-noise epoch lies roughly two Gaussian internal radii away from its matched G1 centroid.

Many epochs lie five to ten Gaussian radii away.

This strongly suggests that the observed latent-domain shift is not produced by a small number of pathological segments.

Instead, the difference appears persistent across the studied off-source region.

---

## 6. Frozen-M10 predictions degrade strongly in real off-source noise

The true physical source used throughout the experiment is approximately

$$
\mathcal M_{\rm true}
\simeq
26.12\,M_\odot,
$$

$$
M_{{\rm tot,true}}
=
60\,M_\odot,
$$

and

$$
\chi_{{\rm eff,true}}
=
0.
$$

### Gaussian empirical-PSD domain G1

The frozen M10 predictor remains reasonably compatible with the injected source.

For chirp mass,

$$
\langle\hat{\mathcal M}\rangle_{G1}
\simeq
26.97\,M_\odot,
$$

with

$$
{\rm bias}
\simeq
+0.85\,M_\odot,
$$

$$
{\rm RMSE}
\simeq
2.28\,M_\odot.
$$

For total mass,

$$
\langle\hat M_{\rm tot}\rangle_{G1}
\simeq
64.65\,M_\odot,
$$

with

$$
{\rm bias}
\simeq
+4.65\,M_\odot,
$$

$$
{\rm RMSE}
\simeq
6.91\,M_\odot.
$$

For effective spin,

$$
\langle\hat\chi_{\rm eff}\rangle_{G1}
\simeq
0.075,
$$

with

$$
{\rm RMSE}
\simeq
0.174.
$$

These results are broadly consistent with the Gaussian-noise behaviour observed in previous M11 experiments.

### Real off-source domain R

The same frozen predictor behaves very differently in real strain.

For chirp mass,

$$
\langle\hat{\mathcal M}\rangle_R
\simeq
17.42\,M_\odot,
$$

with

$$
{\rm bias}
\simeq
-8.70\,M_\odot,
$$

and

$$
{\rm RMSE}
\simeq
10.70\,M_\odot.
$$

For total mass,

$$
\langle\hat M_{\rm tot}\rangle_R
\simeq
43.24\,M_\odot,
$$

with

$$
{\rm bias}
\simeq
-16.76\,M_\odot,
$$

and

$$
{\rm RMSE}
\simeq
22.21\,M_\odot.
$$

For effective spin,

$$
\langle\hat\chi_{\rm eff}\rangle_R
\simeq
-0.305,
$$

with

$$
{\rm RMSE}
\simeq
0.353.
$$

Thus real off-source strain produces a large systematic degradation of the frozen M10 predictor.

The direction of the shift is coherent across targets:

$$
\hat{\mathcal M}_R
<
\hat{\mathcal M}_{G1},
$$

$$
\hat M_{{\rm tot},R}
<
\hat M_{{\rm tot},G1},
$$

and

$$
\hat\chi_{{\rm eff},R}
<
\hat\chi_{{\rm eff},G1}.
$$

---

## 7. Prediction distributions are strongly separated

The G1 and R prediction distributions are clearly distinct.

The KS statistics are approximately

$$
KS
\simeq
0.78
$$

for chirp mass,

$$
KS
\simeq
0.78
$$

for total mass,

and

$$
KS
\simeq
0.74
$$

for $\chi_{\rm eff}$.

The corresponding p-values are extremely small.

The Wasserstein distances are approximately

$$
W_1(\mathcal M)
\simeq
9.55\,M_\odot,
$$

$$
W_1(M_{\rm tot})
\simeq
21.40\,M_\odot,
$$

and

$$
W_1(\chi_{\rm eff})
\simeq
0.381.
$$

The mean domain shifts,

$$
\langle\hat y_R\rangle
-
\langle\hat y_{G1}\rangle,
$$

are correspondingly

$$
-9.55\,M_\odot
$$

for chirp mass,

$$
-21.40\,M_\odot
$$

for total mass,

and

$$
-0.381
$$

for $\chi_{\rm eff}$.

Therefore the domain shift is not a small fluctuation around the same prediction distribution.

---

## 8. Epoch-matched prediction analysis confirms the same conclusion

For each real epoch, the R prediction was compared with the mean of the three G1 controls generated for the same source geometry and GPS time.

The mean epoch-matched shifts are

$$
\langle
\hat{\mathcal M}_R
-
\langle\hat{\mathcal M}_{G1}\rangle
\rangle
\simeq
-9.55\,M_\odot,
$$

$$
\langle
\hat M_{{\rm tot},R}
-
\langle\hat M_{{\rm tot},G1}\rangle
\rangle
\simeq
-21.40\,M_\odot,
$$

and

$$
\langle
\hat\chi_{{\rm eff},R}
-
\langle\hat\chi_{{\rm eff},G1}\rangle
\rangle
\simeq
-0.381.
$$

The median absolute shifts are approximately

$$
8.63\,M_\odot,
$$

$$
19.77\,M_\odot,
$$

and

$$
0.366,
$$

respectively.

Relative to the small Gaussian prediction cloud at each epoch, the median absolute standardized shifts are approximately

$$
3.53
$$

for chirp mass,

$$
3.21
$$

for total mass,

and

$$
2.41
$$

for $\chi_{\rm eff}$.

Although only three G1 controls are available per epoch and these standardized quantities should not be interpreted as rigorously calibrated z-scores, they provide a useful diagnostic scale.

The real-noise predictions lie systematically outside the local Gaussian variability expected for the same source and epoch.

---

## 9. The effect is not dominated by a small number of extreme epochs

The largest CNN residual norms are distributed across multiple epochs rather than being concentrated in one isolated segment.

Likewise, large latent-domain displacements occur repeatedly across the real-noise region.

Several epochs exhibit relative embedding shifts of order

$$
4-10
$$

times the local G1 internal radius, with the largest reaching approximately

$$
11.6.
$$

This argues against interpreting the G1-to-R difference as the consequence of one single strong glitch or pathological segment.

Instead, the current evidence is more consistent with a persistent detector-domain mismatch across the tested region.

---

## 10. Realized SNR explains only part of the real-noise prediction error

Within the R domain, the relationship between

$$
\Delta\rho
$$

and prediction residuals is moderate for the mass parameters.

For chirp mass,

$$
r_{\rm Pearson}
\simeq
0.386,
$$

and

$$
\rho_{\rm Spearman}
\simeq
0.389.
$$

For total mass,

$$
r_{\rm Pearson}
\simeq
0.398,
$$

and

$$
\rho_{\rm Spearman}
\simeq
0.403.
$$

The corresponding p-values are approximately

$$
0.03.
$$

For $\chi_{\rm eff}$, no comparable association is observed.

The mass correlations are stronger than those found in the controlled Gaussian M11.3 experiment.

Nevertheless,

$$
r^2
\sim
0.15-0.16,
$$

so realized SNR still explains only a limited fraction of the prediction variance.

Therefore the real-noise domain shift cannot be reduced to a one-dimensional SNR effect.

---

# Overall M11.5-B interpretation

M11.5-B provides strong evidence that

$$
\boxed{
\text{Gaussian noise matched only through an empirical PSD is not sufficient to reproduce the real off-source detector domain seen by M10.}
}
$$

The difference appears consistently at three levels:

$$
\text{processed strain}
$$

$$
\Downarrow
$$

$$
\text{latent representation}
$$

$$
\Downarrow
$$

$$
\text{parameter prediction}.
$$

The empirical PSD successfully captures the second-order spectral sensitivity of the detector, but the actual real strain contains additional statistical structure that survives whitening and per-detector z-score normalization.

This additional structure produces:

- strong detector-dependent processed-strain differences, especially in L1 and V1;
- a large and systematic latent-space displacement;
- increased latent dispersion;
- large biases and prediction variance in all three regression targets;
- persistent epoch-matched shifts that cannot be explained by ordinary Gaussian realization variability alone.

This result directly supports the central M11 hypothesis that real off-source noise contains information not represented by stationary PSD-colored Gaussian simulations. The original M11 roadmap explicitly identified non-stationarity, glitches, instrumental lines and real detector-sensitivity structure as missing ingredients of M10 Gaussian noise, and proposed synthetic GW injections over real off-source strain as the priority next step. :contentReference[oaicite:0]{index=0}

However, two technical questions remain open before M11.5 can be considered fully closed:

1. the G1 realized-SNR statistic must be corrected so that the Gaussian control again reproduces

$$
\Delta\rho\sim\mathcal N(0,1);
$$

2. the very large L1/V1 processed-scale mismatch must be characterized further to distinguish broad PSD mismatch from genuinely non-Gaussian/non-stationary real-noise structure.

The current evidence is therefore sufficient to establish a substantial G1-to-R domain shift, but not yet sufficient to identify its complete physical/statistical origin.

# M11.5-C — Real-noise residual characterization

M11.5-B produced strong evidence that actual off-source strain $R$ differs from the PSD-matched Gaussian domain $G1$ at the processed-input, latent-representation and prediction levels.

However, two observations require additional investigation before interpreting the result physically.

First, the Gaussian realized-SNR control failed dramatically:

$$
\mathrm{Std}(\Delta\rho_{G1}) \gg 1,
$$

despite the same Gaussian generator having passed the M11.4 normalization tests.

This indicates a technical inconsistency in the way the M11.5 G1 processing-context realization is converted into the 4 s matched-filter statistic.

Second, the processed real strain showed very large detector-dependent scale differences, particularly for L1 and V1.

Before attributing these differences to non-Gaussian or non-stationary detector structure, we must determine whether they can instead be explained by a mismatch between the PSD window used for whitening and the actual noise region being analysed.

M11.5-C therefore addresses three questions:

1. Why does the G1 realized-SNR control fail?
2. Can the L1/V1 scale discrepancy be explained by PSD drift?
3. After controlling for PSD scale, does real strain retain non-Gaussian structure that survives M10 preprocessing?

In [ ]:
delta_G1_B_original = np.asarray(
    delta_G1_B,
    dtype=float,
).copy()

delta_R_B_original = np.asarray(
    delta_R_B,
    dtype=float,
).copy()

print(
    "Original G1:",
    np.mean(delta_G1_B_original),
    np.std(delta_G1_B_original, ddof=1),
)

print(
    "Original R:",
    np.mean(delta_R_B_original),
    np.std(delta_R_B_original, ddof=1),
)

## C1.1 — Processing-grid versus 4 s PSD

The same empirical off-source strain is converted into two discrete PSD representations:

$$
S_n^{\rm proc}(f)
$$

for the longer processing context, and

$$
S_n^{4s}(f)
$$

for the final 4 s matched-filter interval.

If these discretizations are consistent, interpolating the processing PSD onto the 4 s frequency grid should produce ratios close to unity over the analysis band.

In [ ]:
from scipy.interpolate import interp1d

rows = []

for ifo in DETECTORS:

    p_proc = empirical_psds_processing[ifo]
    p_4s = empirical_psds_4s[ifo]

    f_proc = p_proc.sample_frequencies.numpy()
    f_4s = p_4s.sample_frequencies.numpy()

    a_proc = p_proc.numpy()
    a_4s = p_4s.numpy()

    interp_proc = interp1d(
        f_proc,
        a_proc,
        kind="linear",
        bounds_error=False,
        fill_value=np.nan,
    )

    proc_on_4s = interp_proc(f_4s)

    valid = (
        (f_4s >= config.low_frequency_cutoff)
        & (f_4s <= 512.0)
        & np.isfinite(proc_on_4s)
        & np.isfinite(a_4s)
        & (proc_on_4s > 0)
        & (a_4s > 0)
    )

    ratio = (
        proc_on_4s[valid]
        / a_4s[valid]
    )

    rows.append({
        "ifo": ifo,
        "median_ratio":
            np.median(ratio),
        "q05_ratio":
            np.quantile(ratio, 0.05),
        "q95_ratio":
            np.quantile(ratio, 0.95),
        "max_ratio":
            np.max(ratio),
        "min_ratio":
            np.min(ratio),
    })

df_C1_psd_grid_consistency = pd.DataFrame(
    rows
)

df_C1_psd_grid_consistency

In [ ]:
for ifo in DETECTORS:

    p_proc = empirical_psds_processing[ifo]
    p_4s = empirical_psds_4s[ifo]

    f_proc = p_proc.sample_frequencies.numpy()
    f_4s = p_4s.sample_frequencies.numpy()

    interp_proc = interp1d(
        f_proc,
        p_proc.numpy(),
        bounds_error=False,
        fill_value=np.nan,
    )

    ratio = (
        interp_proc(f_4s)
        / p_4s.numpy()
    )

    valid = (
        (f_4s >= 30)
        & (f_4s <= 512)
        & np.isfinite(ratio)
        & (ratio > 0)
    )

    plt.figure(figsize=(9, 4))

    plt.semilogy(
        f_4s[valid],
        ratio[valid],
    )

    plt.axhline(
        1.0,
        linestyle="--",
    )

    plt.xlabel("Frequency [Hz]")
    plt.ylabel(
        r"$S_n^{proc}/S_n^{4s}$"
    )

    plt.title(
        f"{ifo}: PSD discretization consistency"
    )

    plt.show()

## C1.2 — Direct 4 s Gaussian control

The first diagnostic reproduces the ideal matched-filter contract used in M11.4.

Gaussian noise is generated directly on the same 4 s Fourier grid and with the same PSD used by the realized-SNR statistic.

If the Gaussian generator and inner-product implementation are correct, then

$$
\Delta\rho
=
\rho_{\rm real}
-
\rho_{\rm opt}
$$

must recover approximately

$$
\mathcal N(0,1).
$$

Failure here would indicate a fundamental problem in the generator or matched-filter implementation.

Success here, combined with failure after processing-context generation and cropping, would localize the problem to the finite-context/cropping construction.

In [ ]:
network_C1 = R_samples_B[0]["network"]

print(
    "reference epoch:",
    R_samples_B[0]["center_time"],
)

In [ ]:
def generate_direct_4s_gaussian_noise(
    psds,
    seed,
):
    noises = {}

    output_start = float(
        network_C1
        .placement
        .segment_start_time
    )

    for k, ifo in enumerate(DETECTORS):

        detector_seed = (
            seed
            + 10_000 * k
        )

        z = generate_white_frequency_draw(
            flength=len(psds[ifo]),
            seed=detector_seed,
        )

        nf = color_frequency_draw_with_psd(
            z=z,
            psd=psds[ifo],
        )

        nt = nf.to_timeseries()

        nt = injector.set_strain_start_time(
            strain=nt,
            start_time=output_start,
            expected_length=config.length,
        )

        noises[ifo] = nt

    return noises

In [ ]:
#Emsemble

N_C1 = 200
C1_SEED0 = 700_000

delta_direct_4s = []

for j in range(N_C1):

    noises = (
        generate_direct_4s_gaussian_noise(
            psds=empirical_psds_4s,
            seed=C1_SEED0 + j,
        )
    )

    data = {
        ifo:
            noises[ifo]
            + network_C1.signal_segments[ifo]
        for ifo in DETECTORS
    }

    snr = (
        compute_signed_network_realized_snr(
            data_segments=data,
            signal_segments=(
                network_C1.signal_segments
            ),
            psds=empirical_psds_4s,
        )
    )

    delta_direct_4s.append(
        snr["delta_rho_network"]
    )

delta_direct_4s = np.asarray(
    delta_direct_4s
)

In [ ]:
print(
    "mean:",
    delta_direct_4s.mean(),
)

print(
    "std:",
    delta_direct_4s.std(
        ddof=1
    ),
)

print(
    "q05:",
    np.quantile(
        delta_direct_4s,
        0.05,
    ),
)

print(
    "q95:",
    np.quantile(
        delta_direct_4s,
        0.95,
    ),
)

In [ ]:
def build_signal_processing_context(
    network,
):
    output_start = float(
        network
        .placement
        .segment_start_time
    )

    context_start = (
        output_start
        - config.processing_context_start_seconds
    )

    signal_context = {}

    for ifo in DETECTORS:

        zero = injector.build_zero_strain(
            start_time=context_start,
            length=config.processing_length,
        )

        result = injector.inject(
            strain=zero,
            signal=(
                network
                .projection
                .strains[ifo]
            ),
        )

        signal_context[ifo] = (
            result.strain
        )

    return signal_context

In [ ]:
signal_context_C1 = (
    build_signal_processing_context(
        network_C1
    )
)

for ifo in DETECTORS:
    print(
        ifo,
        len(signal_context_C1[ifo]),
        signal_context_C1[ifo].start_time,
    )

In [ ]:
def compute_signed_network_realized_snr_general(
    data_segments,
    signal_segments,
    psds,
):
    dh_sum = 0.0
    hh_sum = 0.0

    for ifo in DETECTORS:

        dh = noise_weighted_inner_product(
            data_segments[ifo],
            signal_segments[ifo],
            psds[ifo],
            config.low_frequency_cutoff,
        )

        hh = noise_weighted_inner_product(
            signal_segments[ifo],
            signal_segments[ifo],
            psds[ifo],
            config.low_frequency_cutoff,
        )

        dh_sum += dh
        hh_sum += hh

    rho_opt = np.sqrt(hh_sum)

    rho_real = (
        dh_sum
        / rho_opt
    )

    return {
        "rho_opt": rho_opt,
        "rho_real": rho_real,
        "delta_rho":
            rho_real - rho_opt,
    }

In [ ]:
def generate_processing_gaussian_noise_C1(
    seed,
):
    output_start = float(
        network_C1
        .placement
        .segment_start_time
    )

    context_start = (
        output_start
        - config.processing_context_start_seconds
    )

    noises = {}

    for k, ifo in enumerate(DETECTORS):

        detector_seed = (
            seed
            + 10_000 * k
        )

        z = (
            generate_white_frequency_draw(
                flength=len(
                    empirical_psds_processing[
                        ifo
                    ]
                ),
                seed=detector_seed,
            )
        )

        nf = color_frequency_draw_with_psd(
            z=z,
            psd=(
                empirical_psds_processing[
                    ifo
                ]
            ),
        )

        nt = nf.to_timeseries()

        nt = injector.set_strain_start_time(
            strain=nt,
            start_time=context_start,
            expected_length=(
                config.processing_length
            ),
        )

        noises[ifo] = nt

    return noises

In [ ]:
delta_long_context = []

for j in range(N_C1):

    noises = (
        generate_processing_gaussian_noise_C1(
            C1_SEED0 + j
        )
    )

    data = {
        ifo:
            noises[ifo]
            + signal_context_C1[ifo]
        for ifo in DETECTORS
    }

    snr = (
        compute_signed_network_realized_snr_general(
            data_segments=data,
            signal_segments=(
                signal_context_C1
            ),
            psds=(
                empirical_psds_processing
            ),
        )
    )

    delta_long_context.append(
        snr["delta_rho"]
    )

delta_long_context = np.asarray(
    delta_long_context
)

In [ ]:
print(
    "long-context mean:",
    delta_long_context.mean(),
)

print(
    "long-context std:",
    delta_long_context.std(
        ddof=1
    ),
)

print(
    "q05:",
    np.quantile(
        delta_long_context,
        0.05,
    ),
)

print(
    "q95:",
    np.quantile(
        delta_long_context,
        0.95,
    ),
)

In [ ]:
delta_long_then_crop = []

for j in range(N_C1):

    noises_long = (
        generate_processing_gaussian_noise_C1(
            C1_SEED0 + j
        )
    )

    output_start = float(
        network_C1
        .placement
        .segment_start_time
    )

    output_end = (
        output_start
        + config.duration
    )

    noise_4s = {
        ifo:
            noises_long[ifo].time_slice(
                output_start,
                output_end,
            )
        for ifo in DETECTORS
    }

    data_4s = {
        ifo:
            noise_4s[ifo]
            + network_C1.signal_segments[ifo]
        for ifo in DETECTORS
    }

    snr = (
        compute_signed_network_realized_snr(
            data_segments=data_4s,
            signal_segments=(
                network_C1.signal_segments
            ),
            psds=empirical_psds_4s,
        )
    )

    delta_long_then_crop.append(
        snr["delta_rho_network"]
    )

delta_long_then_crop = np.asarray(
    delta_long_then_crop
)

In [ ]:
print(
    "long -> crop mean:",
    delta_long_then_crop.mean(),
)

print(
    "long -> crop std:",
    delta_long_then_crop.std(
        ddof=1
    ),
)

print(
    "q05:",
    np.quantile(
        delta_long_then_crop,
        0.05,
    ),
)

print(
    "q95:",
    np.quantile(
        delta_long_then_crop,
        0.95,
    ),
)

In [ ]:
df_C1_delta_summary = pd.DataFrame([
    {
        "construction":
            "direct_4s",
        **summarize_delta_rho(
            delta_direct_4s
        ),
    },
    {
        "construction":
            "direct_processing_context",
        **summarize_delta_rho(
            delta_long_context
        ),
    },
    {
        "construction":
            "processing_context_then_crop",
        **summarize_delta_rho(
            delta_long_then_crop
        ),
    },
])

df_C1_delta_summary

In [ ]:
N_PSD_C1 = 100

cropped_noise_C1 = {
    ifo: []
    for ifo in DETECTORS
}

for j in range(N_PSD_C1):

    noises_long = (
        generate_processing_gaussian_noise_C1(
            900_000 + j
        )
    )

    output_start = float(
        network_C1
        .placement
        .segment_start_time
    )

    output_end = (
        output_start
        + config.duration
    )

    for ifo in DETECTORS:

        crop = noises_long[
            ifo
        ].time_slice(
            output_start,
            output_end,
        )

        cropped_noise_C1[
            ifo
        ].append(
            crop
        )

In [ ]:
periodogram_mean_C1 = {}

for ifo in DETECTORS:

    ps = []

    for ts in cropped_noise_C1[
        ifo
    ]:

        fs = (
            ts.to_frequencyseries()
        )

        p = (
            2.0
            * config.delta_t
            / len(ts)
            * np.abs(
                fs.numpy()
            )**2
        )

        ps.append(p)

    periodogram_mean_C1[
        ifo
    ] = np.mean(
        ps,
        axis=0,
    )

In [ ]:
for ifo in DETECTORS:

    f = (
        empirical_psds_4s[
            ifo
        ]
        .sample_frequencies
        .numpy()
    )

    target = (
        empirical_psds_4s[
            ifo
        ].numpy()
    )

    recovered = (
        periodogram_mean_C1[
            ifo
        ]
    )

    valid = (
        (f >= 30)
        & (f <= 512)
        & (target > 0)
        & np.isfinite(target)
        & np.isfinite(recovered)
    )

    ratio = (
        recovered[valid]
        / target[valid]
    )

    ratio_norm = (
        ratio
        / np.median(ratio)
    )

    plt.figure(figsize=(9, 4))

    plt.semilogy(
        f[valid],
        ratio_norm,
    )

    plt.axhline(
        1.0,
        linestyle="--",
    )

    plt.xlabel("Frequency [Hz]")
    plt.ylabel(
        "Recovered / target "
        "(median normalized)"
    )

    plt.title(
        f"{ifo}: spectral distortion after long-context crop"
    )

    plt.show()

### C1 decision logic

The realized-SNR issue will be localized according to the following hierarchy.

If direct 4 s Gaussian noise fails,

$$
\Rightarrow
$$

the generator, PSD normalization or inner-product implementation is inconsistent.

If direct 4 s passes but full processing-context noise fails,

$$
\Rightarrow
$$

the processing-length Gaussian construction is inconsistent.

If both direct constructions pass but

$$
\text{processing context}
\rightarrow
\text{4 s crop}
$$

fails,

$$
\Rightarrow
$$

the problem is specifically caused by the finite-context cropping operation and its effective spectral statistics.

No change to the production pipeline will be made until this hierarchy identifies the failing step.

# C2 — PSD mismatch between the whitening reference and real-noise region

M11.5-B whitened all real-noise samples using the fixed W0 PSD estimated from

$$
[-1024,-640]\ {\rm s}.
$$

The real-noise samples themselves were extracted from

$$
[-512,-128]\ {\rm s}.
$$

The large residual processed scales observed for L1 and V1 could therefore arise if the detector PSD changed substantially between these intervals.

We now estimate a second empirical PSD directly from the real-noise region and compare it against W0.

In [ ]:
PSD_WINDOW_R_LOCAL = (
    -512.0,
    -128.0,
)

In [ ]:
empirical_psds_Rlocal_processing = {}

for ifo in DETECTORS:

    empirical_psds_Rlocal_processing[
        ifo
    ] = estimate_offsource_psd(
        strain=raw_strains_long[ifo],
        event_time=EVENT_TIME,
        delta_f=(
            config.processing_delta_f
        ),
        sampling_frequency=(
            config.sampling_frequency
        ),
        target_flength=(
            config.processing_flength
        ),
        psd_start_offset=(
            PSD_WINDOW_R_LOCAL[0]
        ),
        psd_end_offset=(
            PSD_WINDOW_R_LOCAL[1]
        ),
        psd_segment_duration=(
            PSD_SEGMENT_DURATION_R
        ),
        low_frequency_cutoff=(
            config.low_frequency_cutoff
        ),
        max_filter_duration=0.5,
        trunc_method="hann",
    )

In [ ]:
BANDS_C2 = [
    (30, 60),
    (60, 120),
    (120, 256),
    (256, 512),
]

rows = []

for ifo in DETECTORS:

    p0 = (
        empirical_psds_processing[
            ifo
        ]
    )

    p1 = (
        empirical_psds_Rlocal_processing[
            ifo
        ]
    )

    f = (
        p0
        .sample_frequencies
        .numpy()
    )

    a0 = p0.numpy()
    a1 = p1.numpy()

    for fmin, fmax in BANDS_C2:

        mask = (
            (f >= fmin)
            & (f < fmax)
            & (a0 > 0)
            & (a1 > 0)
            & np.isfinite(a0)
            & np.isfinite(a1)
        )

        ratio = (
            a1[mask]
            / a0[mask]
        )

        rows.append({
            "ifo": ifo,
            "fmin": fmin,
            "fmax": fmax,
            "median_ratio":
                np.median(ratio),
            "q10_ratio":
                np.quantile(
                    ratio,
                    0.10,
                ),
            "q90_ratio":
                np.quantile(
                    ratio,
                    0.90,
                ),
        })

df_C2_psd_ratios = pd.DataFrame(
    rows
)

df_C2_psd_ratios

In [ ]:
for ifo in DETECTORS:

    p0 = (
        empirical_psds_processing[
            ifo
        ]
    )

    p1 = (
        empirical_psds_Rlocal_processing[
            ifo
        ]
    )

    f = (
        p0
        .sample_frequencies
        .numpy()
    )

    ratio = (
        p1.numpy()
        / p0.numpy()
    )

    valid = (
        (f >= 30)
        & (f <= 512)
        & np.isfinite(ratio)
        & (ratio > 0)
    )

    plt.figure(figsize=(9, 4))

    plt.semilogy(
        f[valid],
        ratio[valid],
    )

    plt.axhline(
        1.0,
        linestyle="--",
    )

    plt.xlabel("Frequency [Hz]")
    plt.ylabel(
        r"$S_n^{R-region}/S_n^{W0}$"
    )

    plt.title(
        f"{ifo}: local PSD drift"
    )

    plt.show()

In [ ]:
#Reprocesamos los mismos R samples con la PSD local

def reprocess_R_sample_with_psds(
    r_sample,
    psds,
):
    strains = {
        ifo:
            r_sample[
                "injection"
            ][ifo].strain
        for ifo in DETECTORS
    }

    processed = (
        processor.process_network(
            strains=strains,
            psds=psds,
        )
    )

    X = np.stack(
        [
            np.asarray(
                processed[ifo],
                dtype=np.float64,
            )
            for ifo in DETECTORS
        ],
        axis=0,
    )

    Z, _, _ = (
        m10_input_zscore(
            X,
            M10_INPUT_NORM_CONFIG,
        )
    )

    return (
        X,
        Z.astype(np.float32),
    )

In [ ]:
X_R_localPSD_C2 = []
Z_R_localPSD_C2 = []

for r in R_samples_B:

    X, Z = (
        reprocess_R_sample_with_psds(
            r_sample=r,
            psds=(
                empirical_psds_Rlocal_processing
            ),
        )
    )

    X_R_localPSD_C2.append(X)
    Z_R_localPSD_C2.append(Z)

X_R_localPSD_C2 = np.stack(
    X_R_localPSD_C2
)

Z_R_localPSD_C2 = np.stack(
    Z_R_localPSD_C2
)

In [ ]:
rows = []

for k, ifo in enumerate(
    DETECTORS
):

    std_W0 = np.asarray([
        np.std(r["X"][k])
        for r in R_samples_B
    ])

    std_local = (
        np.std(
            X_R_localPSD_C2[
                :,
                k,
                :
            ],
            axis=1,
        )
    )

    rows.append({
        "ifo": ifo,

        "mean_std_W0":
            std_W0.mean(),

        "mean_std_local":
            std_local.mean(),

        "ratio_local_over_W0":
            (
                std_local.mean()
                / std_W0.mean()
            ),

        "median_ratio_samplewise":
            np.median(
                std_local
                / std_W0
            ),
    })

df_C2_rewhitening_scale = pd.DataFrame(
    rows
)

df_C2_rewhitening_scale

# C3 — Residual non-Gaussian structure

After investigating PSD consistency and whitening scale, we characterize the shape of the processed strain distributions.

For Gaussian noise,

$$
\mathrm{skewness}\approx0
$$

and

$$
\mathrm{excess\ kurtosis}\approx0.
$$

Skewness and kurtosis are invariant under affine mean/std normalization, so the M10 per-detector z-score cannot remove these properties.

Therefore deviations observed in the normalized CNN inputs represent statistical structure that survives the M10 normalization itself.

In [ ]:
from scipy.stats import (
    skew,
    kurtosis,
)

In [ ]:
rows = []

for r in R_samples_B:

    for k, ifo in enumerate(
        DETECTORS
    ):

        z = r["Z"][k]

        rows.append({
            "domain": "R",
            "epoch_index":
                r["epoch_index"],
            "ifo": ifo,

            "skewness":
                skew(
                    z,
                    bias=False,
                ),

            "excess_kurtosis":
                kurtosis(
                    z,
                    fisher=True,
                    bias=False,
                ),

            "abs_q99":
                np.quantile(
                    np.abs(z),
                    0.99,
                ),

            "abs_q999":
                np.quantile(
                    np.abs(z),
                    0.999,
                ),

            "abs_max":
                np.max(
                    np.abs(z)
                ),
        })


for g in G1_samples_B:

    for k, ifo in enumerate(
        DETECTORS
    ):

        z = g["Z"][k]

        rows.append({
            "domain": "G1",
            "epoch_index":
                g["epoch_index"],
            "ifo": ifo,

            "skewness":
                skew(
                    z,
                    bias=False,
                ),

            "excess_kurtosis":
                kurtosis(
                    z,
                    fisher=True,
                    bias=False,
                ),

            "abs_q99":
                np.quantile(
                    np.abs(z),
                    0.99,
                ),

            "abs_q999":
                np.quantile(
                    np.abs(z),
                    0.999,
                ),

            "abs_max":
                np.max(
                    np.abs(z)
                ),
        })

df_C3_shape = pd.DataFrame(rows)

In [ ]:
df_C3_shape_summary = (
    df_C3_shape
    .groupby(
        [
            "domain",
            "ifo",
        ]
    )
    .agg(
        median_skew=(
            "skewness",
            "median",
        ),

        q90_abs_skew=(
            "skewness",
            lambda x:
                np.quantile(
                    np.abs(x),
                    0.90,
                ),
        ),

        median_excess_kurtosis=(
            "excess_kurtosis",
            "median",
        ),

        q90_excess_kurtosis=(
            "excess_kurtosis",
            lambda x:
                np.quantile(
                    x,
                    0.90,
                ),
        ),

        median_abs_q99=(
            "abs_q99",
            "median",
        ),

        median_abs_q999=(
            "abs_q999",
            "median",
        ),

        median_abs_max=(
            "abs_max",
            "median",
        ),
    )
    .reset_index()
)

df_C3_shape_summary

In [ ]:
# Test G1 vs R

rows = []

for ifo in DETECTORS:

    for metric in [
        "skewness",
        "excess_kurtosis",
        "abs_q99",
        "abs_q999",
        "abs_max",
    ]:

        g = (
            df_C3_shape
            .query(
                "domain == 'G1' "
                "and ifo == @ifo"
            )[metric]
            .to_numpy()
        )

        r = (
            df_C3_shape
            .query(
                "domain == 'R' "
                "and ifo == @ifo"
            )[metric]
            .to_numpy()
        )

        ks = ks_2samp(
            g,
            r,
        )

        rows.append({
            "ifo": ifo,
            "metric": metric,

            "median_G1":
                np.median(g),

            "median_R":
                np.median(r),

            "KS":
                ks.statistic,

            "KS_p":
                ks.pvalue,

            "Wasserstein":
                wasserstein_distance(
                    g,
                    r,
                ),
        })

df_C3_shape_tests = pd.DataFrame(
    rows
)

df_C3_shape_tests

In [ ]:
for ifo in DETECTORS:

    dR = (
        df_C3_shape
        .query(
            "domain == 'R' "
            "and ifo == @ifo"
        )
        .sort_values(
            "epoch_index"
        )
    )

    dG = (
        df_C3_shape
        .query(
            "domain == 'G1' "
            "and ifo == @ifo"
        )
        .groupby(
            "epoch_index"
        )[
            "excess_kurtosis"
        ]
        .mean()
        .reset_index()
    )

    plt.figure(
        figsize=(9, 4)
    )

    plt.plot(
        dR["epoch_index"],
        dR[
            "excess_kurtosis"
        ],
        marker="o",
        label="R",
    )

    plt.plot(
        dG["epoch_index"],
        dG[
            "excess_kurtosis"
        ],
        marker="o",
        label="mean G1",
    )

    plt.axhline(
        0.0,
        linestyle="--",
    )

    plt.xlabel(
        "Epoch index"
    )

    plt.ylabel(
        "Excess kurtosis"
    )

    plt.title(
        f"{ifo}: non-Gaussian tail structure"
    )

    plt.legend()
    plt.show()

In [ ]:
# Real noise dependance with latent displacement

df_C3_R_epoch = (
    df_C3_shape
    .query(
        "domain == 'R'"
    )
    .groupby(
        "epoch_index"
    )
    .agg(
        mean_abs_skew=(
            "skewness",
            lambda x:
                np.mean(
                    np.abs(x)
                ),
        ),

        mean_excess_kurtosis=(
            "excess_kurtosis",
            "mean",
        ),

        max_excess_kurtosis=(
            "excess_kurtosis",
            "max",
        ),

        mean_abs_q999=(
            "abs_q999",
            "mean",
        ),
    )
    .reset_index()
)

In [ ]:
df_C3_shape_embedding = (
    df_C3_R_epoch
    .merge(
        df_epoch_embedding_B[
            [
                "epoch_index",
                "relative_epoch_embedding_shift",
            ]
        ],
        on="epoch_index",
        how="left",
    )
)

In [ ]:
rows = []

for metric in [
    "mean_abs_skew",
    "mean_excess_kurtosis",
    "max_excess_kurtosis",
    "mean_abs_q999",
]:

    x = (
        df_C3_shape_embedding[
            metric
        ].to_numpy()
    )

    y = (
        df_C3_shape_embedding[
            "relative_epoch_embedding_shift"
        ].to_numpy()
    )

    p = pearsonr(x, y)
    s = spearmanr(x, y)

    rows.append({
        "metric": metric,

        "pearson_r":
            p.statistic,

        "pearson_p":
            p.pvalue,

        "spearman_rho":
            s.statistic,

        "spearman_p":
            s.pvalue,
    })

df_C3_shape_embedding_corr = (
    pd.DataFrame(rows)
)

df_C3_shape_embedding_corr

In [ ]:
# Now shape with prediction error

cnn_residual_norm_C3 = (
    df_epoch_diagnostic_B[
        [
            "epoch_index",
            "cnn_residual_norm",
        ]
    ]
)

In [ ]:
df_C3_shape_prediction = (
    df_C3_R_epoch
    .merge(
        cnn_residual_norm_C3,
        on="epoch_index",
        how="left",
    )
)

In [ ]:
rows = []

for metric in [
    "mean_abs_skew",
    "mean_excess_kurtosis",
    "max_excess_kurtosis",
    "mean_abs_q999",
]:

    x = (
        df_C3_shape_prediction[
            metric
        ].to_numpy()
    )

    y = (
        df_C3_shape_prediction[
            "cnn_residual_norm"
        ].to_numpy()
    )

    p = pearsonr(x, y)
    s = spearmanr(x, y)

    rows.append({
        "metric": metric,

        "pearson_r":
            p.statistic,

        "pearson_p":
            p.pvalue,

        "spearman_rho":
            s.statistic,

        "spearman_p":
            s.pvalue,
    })

df_C3_shape_prediction_corr = (
    pd.DataFrame(rows)
)

df_C3_shape_prediction_corr

## C4 — Noise-only processed control

Hay una última comprobación que considero importante.

En B y C3 estamos caracterizando:

$$ h+n. $$

Como la señal es idéntica conceptualmente entre G1/R y tiene SNR 15, el contraste sigue siendo válido.

Pero para afirmar estrictamente:

“el noise real tiene esta shape”

es mejor mirar también:

$$ n $$

sin GW.

In [ ]:
X_R_noiseonly_C4 = []

for r in R_samples_B:

    noises = (
        extract_real_processing_noise(
            raw_strains_long,
            r["network"],
        )
    )

    processed = (
        processor.process_network(
            strains=noises,
            psds=empirical_psds_processing,
        )
    )

    X = np.stack(
        [
            np.asarray(
                processed[ifo],
                dtype=np.float64,
            )
            for ifo in DETECTORS
        ],
        axis=0,
    )

    X_R_noiseonly_C4.append(X)

X_R_noiseonly_C4 = np.stack(
    X_R_noiseonly_C4
)

In [ ]:
X_G1_noiseonly_C4 = []

for g in G1_samples_B:

    epoch_index = (
        g["epoch_index"]
    )

    network = (
        R_samples_B[
            epoch_index
        ]["network"]
    )

    context_start = (
        float(
            network
            .placement
            .segment_start_time
        )
        - config.processing_context_start_seconds
    )

    noises = {}

    for k, ifo in enumerate(
        DETECTORS
    ):

        detector_seed = (
            g["gaussian_seed"]
            + 10_000 * k
        )

        z = (
            generate_white_frequency_draw(
                flength=len(
                    empirical_psds_processing[
                        ifo
                    ]
                ),
                seed=detector_seed,
            )
        )

        nf = (
            color_frequency_draw_with_psd(
                z=z,
                psd=(
                    empirical_psds_processing[
                        ifo
                    ]
                ),
            )
        )

        nt = nf.to_timeseries()

        noises[ifo] = (
            injector.set_strain_start_time(
                strain=nt,
                start_time=context_start,
                expected_length=(
                    config.processing_length
                ),
            )
        )

    processed = (
        processor.process_network(
            strains=noises,
            psds=empirical_psds_processing,
        )
    )

    X = np.stack(
        [
            np.asarray(
                processed[ifo],
                dtype=np.float64,
            )
            for ifo in DETECTORS
        ],
        axis=0,
    )

    X_G1_noiseonly_C4.append(X)

X_G1_noiseonly_C4 = np.stack(
    X_G1_noiseonly_C4
)

In [ ]:
rows = []

for domain, X_all in [
    (
        "G1",
        X_G1_noiseonly_C4,
    ),
    (
        "R",
        X_R_noiseonly_C4,
    ),
]:

    for k, ifo in enumerate(
        DETECTORS
    ):

        for x in X_all[:, k, :]:

            z = (
                x - np.mean(x)
            ) / np.std(x)

            rows.append({
                "domain": domain,
                "ifo": ifo,

                "raw_std":
                    np.std(x),

                "skewness":
                    skew(
                        z,
                        bias=False,
                    ),

                "excess_kurtosis":
                    kurtosis(
                        z,
                        fisher=True,
                        bias=False,
                    ),

                "abs_q999":
                    np.quantile(
                        np.abs(z),
                        0.999,
                    ),
            })

df_C4_noiseonly = pd.DataFrame(
    rows
)

In [ ]:
df_C4_noiseonly_summary = (
    df_C4_noiseonly
    .groupby(
        [
            "domain",
            "ifo",
        ]
    )
    .median(
        numeric_only=True
    )
    .reset_index()
)

df_C4_noiseonly_summary